# БИБЛИОТЕКИ

In [26]:
# @title Запустить ПЕРВОЙ

import sys
import subprocess
import time
import os
from google.colab import userdata, drive
import shutil
from uuid import UUID
import uuid
import pandas as pd
import sqlite3
import xml.etree.ElementTree as ET
import re
import io
import hashlib
from pathlib import Path
from IPython.display import display, HTML, Markdown
import json
import zlib
import base64
import urllib.parse
from datetime import datetime, timezone

In [ ]:
# @title Запустить только после Установки Python-библиотек проекта

from dotenv import load_dotenv
from databases import Database
from sqlalchemy.dialects.postgresql import UUID as PG_UUID
from constants import (
     POSTGRES_CONNECTION_ASYNC_URL,
     POSTGRES_CONNECTION_SYNC_URL,
     LLM_MODEL,
     CHROMA_COLLECTION,
     CHROMA_PATH,
     EMBEDDINGS_MODEL,
     RECORD_MANAGER,
     MAX_CONTEXT_CHARS,
     MAX_INPUT_CHARS,
     DB_PATH
)
from sqlalchemy import (
    create_engine, inspect, MetaData, Table, Column, String,
    DateTime, Text, Integer, Float, func, select
)
import docx
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.indexing import aindex
from langchain_classic.indexes import SQLRecordManager
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate
import requests
from chroma_async import AsyncChroma
import traceback

/tmp/ipykernel_2498/3619126951.py:23: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


# Работа с Git

## Настройка Git

In [27]:
# Настройка пользователя
!git config --global user.email "maxmif1977@gmail.com"
!git config --global user.name "Максим Митющенко"

# Использование токена (PAT) для доступа (безопаснее пароля)
token = userdata.get('GITHUB_TOKEN')
username = "Mityuschenko"
repository = "RAG_BPMN"

# Формируем URL с токеном внутри для автоматической авторизации
remote_url = f"https://{token}@github.com/{username}/{repository}.git"

### Добавляем `файлы и/или папки` в `.gitignore`

Чтобы файл `.env` с чувствительными данными не попал в Git репозиторий, добавим его в `.gitignore`.

In [ ]:
%%writefile .gitignore
.env
*.gsheet
*.gdoc
Project_Backup/
*.pptx
*.ipynb_checkpoints/
*.ipynb.tmp
__pycache__/
*.pyc
*.tmp

Overwriting .gitignore


In [ ]:
# 1. Добавляем изменения
!git add .

# 2. Фиксируем изменения (commit)
!git commit -m "Выпускной вариант"

# 3. Отправляем в GitHub (обычно в ветку main или master)
!git push origin main


fatal: not a git repository (or any parent up to mount point /content)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
fatal: not a git repository (or any parent up to mount point /content)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
fatal: not a git repository (or any parent up to mount point /content)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


## Клонирование репозитория

In [ ]:
# Клонирование репозитория
!git clone {remote_url}

## Запись в репозиторий

In [ ]:
# Переходим в папку проекта
%cd '/content/drive/MyDrive/Colab Notebooks/RAG_BPMN'

/content/drive/MyDrive/Colab Notebooks/RAG_BPMN


### Инициализация Git в существующей папке и привязка к удаленному репозиторию

Если вы хотите превратить существующую папку в Git-репозиторий и связать её с удаленным репозиторием (например, на GitHub), выполните следующие команды.

**Внимание:** Если удаленный репозиторий уже содержит файлы, а ваша локальная папка тоже содержит файлы, вам может потребоваться `git pull origin main --allow-unrelated-histories`, чтобы объединить их, прежде чем вы сможете отправить свои изменения.

In [ ]:
# 1. Инициализируем новый локальный репозиторий Git в текущей папке.
!git init

# 2. (Опционально) Переименовываем основную ветку в 'main' (современная практика).
!git branch -M main

# 3. Привязываем локальный репозиторий к удаленному.
# Используем переменную `remote_url`, определенную ранее в этом блокноте.
!git remote add origin {remote_url}

# 4. Получаем данные из удаленного репозитория. Используем '--allow-unrelated-histories'
# для объединения, если локальная папка содержит файлы, не связанные с удаленным репозиторием.
!git pull origin main --allow-unrelated-histories

# 5. Добавляем все существующие файлы в отслеживание Git.
!git add .

# 6. Фиксируем изменения (первый коммит).
!git commit -m "Initial commit with existing files"

# 7. Отправляем изменения в удаленный репозиторий.
# Флаг '-u' устанавливает upstream-ветку, чтобы последующие 'git push' и 'git pull' работали без указания 'origin main'.
!git push -u origin main

Reinitialized existing Git repository in /content/drive/MyDrive/Colab Notebooks/RAG_BPMN/.git/
error: remote origin already exists.
error: Pulling is not possible because you have unmerged files.
hint: Fix them up in the work tree, and then use 'git add/rm <file>'
hint: as appropriate to mark resolution and make a commit.
fatal: Exiting because of an unresolved conflict.
[main 78f0514] Initial commit with existing files
To https://github.com/Mityuschenko/RAG_BPMN.git
 ! [rejected]        main -> main (fetch first)
error: failed to push some refs to 'https://github.com/Mityuschenko/RAG_BPMN.git'
hint: Updates were rejected because the remote contains work that you do
hint: not have locally. This is usually caused by another repository pushing
hint: to the same ref. You may want to first integrate the remote changes
hint: (e.g., 'git pull ...') before pushing again.
hint: See the 'Note about fast-forwards' in 'git push --help' for details.


# Работа с Google Диск

In [28]:
# 1. Монтируем Диск
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# @title Копирование Проекта в локальную область Colab

# Указываем точные пути без всяких экранирований
source_path = '/content/drive/MyDrive/Colab Notebooks/RAG_BPMN/Project'
destination_path = '/content/Project'

try:
    # Если папка уже частично создалась, удаляем её для чистого копирования
    if os.path.exists(destination_path):
        shutil.rmtree(destination_path)

    # Копируем всю директорию целиком
    shutil.copytree(source_path, destination_path, ignore=shutil.ignore_patterns('*.ipynb'))
    print("✅ Успех! Проект полностью скопирован в локальную область Colab.")

    # Проверяем содержимое
    print("Содержимое папки проекта:", os.listdir(destination_path))
except Exception as e:
    print(f"❌ Ошибка при копировании: {e}")


✅ Успех! Проект полностью скопирован в локальную область Colab.
Содержимое папки проекта: ['requirements.txt', 'data', 'chroma_async.py', '.env', 'constants.py', 'install_ollama.sh', 'prompts']


In [ ]:
# 2. Путь к корню проекта
PROJECT_PATH = "/content/Project"

# 3. Добавляем только корень в пути Python
sys.path.append(PROJECT_PATH)

# 4. Делаем корень рабочей директорией, чтобы constants.py нашел .env рядом с собой
os.chdir(PROJECT_PATH)
print(f"Рабочая директория: {os.getcwd()}")


Рабочая директория: /content/Project


In [ ]:
# @title Установка Python-библиотек проекта
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.0/81.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.6/346.6 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 100.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 682.6/682.6 kB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 107.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 72.0 MB/s eta 0:00:00
   ━━━

# СОХРАНИТЬ И ЗАГРУЗИТЬ

In [ ]:
# @title БД Бэкап

BACKUP_DIR = "/content/drive/MyDrive/Colab Notebooks/RAG_BPMN/Project_Backup"
LOCAL_CHROMA_DIR = "/content/Project/data/vector_store"  # путь к папке с ChromaDB
LOCAL_RECORD_MANAGER_DB = "/content/Project/data/db_file/record_manager_cache.sql"

# Путь к вашему файлу .env
ENV_PATH = "/content/Project/.env"

# Автоматическая загрузка переменных из .env
if os.path.exists(ENV_PATH):
    load_dotenv(ENV_PATH)
    print("✅ Переменные окружения успешно загружены из .env")
else:
    print(f"⚠️ Файл {ENV_PATH} не найден! Проверьте путь к файлу.")

# Получение параметров PostgreSQL из окружения с дефолтными значениями на случай сбоя
PG_USER = os.getenv("POSTGRES_USER", "analyst")
PG_PASSWORD = os.getenv("POSTGRES_PASSWORD", "12345678")
PG_HOST = os.getenv("POSTGRES_HOST", "127.0.0.1")
PG_PORT = os.getenv("POSTGRES_PORT", "5432")
PG_DB = os.getenv("POSTGRES_DB", "bpmn_models")

def make_backup():
    """Сохраняет данные из Postgres, ChromaDB и SQLRecordManager на Google Диск"""
    print("=== ЗАПУСК РЕЗЕРВНОГО КОПИРОВАНИЯ ===")
    os.makedirs(BACKUP_DIR, exist_ok=True)

    # 1. Дамп PostgreSQL в один файл
    print("Сохраняем PostgreSQL...")
    env = os.environ.copy()
    env["PGPASSWORD"] = PG_PASSWORD
    try:
        with open("/content/postgres_backup.sql", "w") as f:
            subprocess.run(
                [
                    "pg_dump",
                    "-h",
                    PG_HOST,
                    "-p",
                    PG_PORT,
                    "-U",
                    PG_USER,
                    "-d",
                    PG_DB,
                ],
                stdout=f,
                env=env,
                check=True,
            )
        shutil.copy(
            "/content/postgres_backup.sql", f"{BACKUP_DIR}/postgres_backup.sql"
        )
        print("✅ Бэкап PostgreSQL успешно сохранен.")
    except Exception as e:
        print(f"❌ Ошибка бэкапа PostgreSQL: {e}")

    # 2. Архивация папки ChromaDB
    print("Сохраняем ChromaDB...")
    if os.path.exists(LOCAL_CHROMA_DIR):
        shutil.make_archive(
            f"{BACKUP_DIR}/chroma_backup", "zip", LOCAL_CHROMA_DIR
        )
        print("✅ Бэкап ChromaDB успешно сохранен.")
    else:
        print(
            "⚠️ Локальная папка ChromaDB не найдена, бэкап Chroma пропущен."
        )

    # 3. Бэкап базы данных SQLRecordManager (SQLite)
    print("Сохраняем кэш SQLRecordManager...")
    if os.path.exists(LOCAL_RECORD_MANAGER_DB):
        shutil.copy(
            LOCAL_RECORD_MANAGER_DB, f"{BACKUP_DIR}/record_manager_cache.sql"
        )
        print("✅ Бэкап базы данных SQLRecordManager успешно сохранен.")
    else:
        print("⚠️ Файл базы SQLRecordManager не найден по указанному пути.")

    print("🎉 Все данные надежно сохранены на Google Диск!")


✅ Переменные окружения успешно загружены из .env


In [ ]:
# @title БД Восстановление

def restore_from_backup_PostgreSQL():
    """Восстанавливает данные с Google Диска в Colab при новом запуске"""
    print("=== ВОССТАНОВЛЕНИЕ ДАННЫХ ИЗ БЭКАПА ===")

    print("Восстанавливаем PostgreSQL...")
    if os.path.exists(f"{BACKUP_DIR}/postgres_backup.sql"):
        shutil.copy(
            f"{BACKUP_DIR}/postgres_backup.sql", "/content/postgres_backup.sql"
        )
        env = os.environ.copy()
        env["PGPASSWORD"] = PG_PASSWORD
        try:
            with open("/content/postgres_backup.sql", "r") as f:
                subprocess.run(
                    [
                        "psql",
                        "-h",
                        PG_HOST,
                        "-p",
                        PG_PORT,
                        "-U",
                        PG_USER,
                        "-d",
                        PG_DB,
                    ],
                    stdin=f,
                    env=env,
                    check=True,
                )
            print("✅ База данных PostgreSQL успешно восстановлена.")
        except Exception as e:
            print(f"❌ Ошибка восстановления PostgreSQL: {e}")
    else:
        print("ℹ️ Файл бэкапа Postgres не найден, создана чистая БД.")

def restore_from_backup():
    """Восстанавливает данные с Google Диска в Colab при новом запуске"""
    print("=== ВОССТАНОВЛЕНИЕ ДАННЫХ ИЗ БЭКАПА ===")

    # 2. Распаковка ChromaDB
    print("Восстанавливаем ChromaDB...")
    if os.path.exists(f"{BACKUP_DIR}/chroma_backup.zip"):
        os.makedirs(LOCAL_CHROMA_DIR, exist_ok=True)
        shutil.unpack_archive(
            f"{BACKUP_DIR}/chroma_backup.zip", LOCAL_CHROMA_DIR, "zip"
        )
        print("✅ Векторная база ChromaDB успешно восстановлена.")
    else:
        print("ℹ️ Бэкап ChromaDB не найден.")

    # 3. Восстановление базы данных SQLRecordManager (SQLite)
    print("Восстанавливаем кэш SQLRecordManager...")
    if os.path.exists(f"{BACKUP_DIR}/record_manager_cache.db"):
        os.makedirs(os.path.dirname(LOCAL_RECORD_MANAGER_DB), exist_ok=True)
        shutil.copy(
            f"{BACKUP_DIR}/record_manager_cache.db", LOCAL_RECORD_MANAGER_DB
        )
        print("✅ База данных SQLRecordManager успешно восстановлена.")
    else:
        print(
            "ℹ️ Бэкап SQLRecordManager не найден, менеджер создаст пустую базу при старте."
        )


In [ ]:
make_backup()

=== ЗАПУСК РЕЗЕРВНОГО КОПИРОВАНИЯ ===
Сохраняем PostgreSQL...
✅ Бэкап PostgreSQL успешно сохранен.
Сохраняем ChromaDB...
✅ Бэкап ChromaDB успешно сохранен.
Сохраняем кэш SQLRecordManager...
✅ Бэкап базы данных SQLRecordManager успешно сохранен.
🎉 Все данные надежно сохранены на Google Диск!


In [ ]:
restore_from_backup_PostgreSQL()

=== ВОССТАНОВЛЕНИЕ ДАННЫХ ИЗ БЭКАПА ===
Восстанавливаем PostgreSQL...
✅ База данных PostgreSQL успешно восстановлена.


In [ ]:
# @title Проект на Google Диск - Сохранение изменений

# --- НАСТРОЙКА ПУТЕЙ ---
source_path = '/content/Project'
destination_path = '/content/drive/MyDrive/Colab Notebooks/RAG_BPMN/Project'

# --- НАСТРОЙКА ИГНОРИРОВАНИЯ ---
# Впишите сюда названия папок, файлов или расширения, которые НЕ нужно сохранять
IGNORE_LIST = [
    '__pycache__',            # Временный кэш скомпилированного Python-кода
    '.pytest_cache',          # Кэш тестов (если используете pytest)
    '.ipynb_checkpoints',     # Скрытые автосохранения блокнотов Jupyter
    '*.log'                   # Расширение файлов логов (пример маски)
]

def custom_ignore(path, names):
    """Функция-фильтр, которая решает, что копировать, а что игнорировать."""
    ignored = []
    for name in names:
        # Проверяем точное совпадение имени файла/папки
        if name in IGNORE_LIST:
            ignored.append(name)
            continue

        # Проверяем совпадение по расширению (например, *.log или *.db)
        for pattern in IGNORE_LIST:
            if pattern.startswith('*') and name.endswith(pattern[1:]):
                ignored.append(name)
                break
    return ignored

# --- ПРОЦЕСС КОПИРОВАНИЯ ---
try:
    # 1. Сначала удаляем старую папку на Диске для чистого обновления
    if os.path.exists(destination_path):
        print("Удаляем старую копию проекта на Диске...")
        shutil.rmtree(destination_path)

    # 2. Копируем проект с применением нашего фильтра игнорирования
    print("Копируем обновленные файлы на Диск...")
    shutil.copytree(source_path, destination_path, ignore=custom_ignore)

    print("✅ Успех! Проект успешно синхронизирован с Google Диском.")
    print("Выделенные файлы и папки кэша были успешно проигнорированы.")

except Exception as e:
    print(f"❌ Ошибка при сохранении: {e}")




Удаляем старую копию проекта на Диске...
Копируем обновленные файлы на Диск...
✅ Успех! Проект успешно синхронизирован с Google Диском.
Выделенные файлы и папки кэша были успешно проигнорированы.


# PostgreSQL

In [ ]:
# @title PostgreSQL - установка

# 1. Импортируем ключ репозитория
!curl -fsSL https://www.postgresql.org/media/keys/ACCC4CF8.asc | sudo gpg --dearmor -o /etc/apt/trusted.gpg.d/apt.postgresql.org.gpg

# 2. Добавляем репозиторий PostgreSQL для текущей версии Ubuntu (jammy)
!echo "deb [arch=amd64,arm64 signed-by=/etc/apt/trusted.gpg.d/apt.postgresql.org.gpg] https://apt.postgresql.org/pub/repos/apt $(lsb_release -cs)-pgdg main" | sudo tee /etc/apt/sources.list.d/pgdg.list > /dev/null

# 3. Обновляем список пакетов
!sudo apt-get update -qq

# 4. Устанавливаем PostgreSQL (рекомендуется использовать последнюю стабильную версию, например, 16)
!sudo apt-get install -y postgresql-16 postgresql-contrib-16

print("PostgreSQL 16 успешно установлен.")

# Проверка установки
!psql --version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Note, selecting 'postgresql-16' instead of 'postgresql-contrib-16'
The following additional packages will be installed:
  libcommon-sense-perl libjson-perl libjson-xs-perl libpq-dev libpq5
  libtypes-serialiser-perl logrotate netbase postgresql-client-16
  postgresql-client-common postgresql-common ssl-cert sysstat
Suggested packages:
  postgresql-doc-18 libpq-oauth bsd-mailx | mailx postgresql-doc-16 isag
The following NEW packages will be installed:
  libcommon-sense-perl libjson-perl libjson-xs-perl libtypes-serialiser-perl
  logrotate netbase postgresql-16 postgresql-client-16
  postgresql-client-common postgresql-common ssl-cert sysstat
The following packages will be upgraded:
  libpq-dev libpq5
2 up

In [ ]:
# @title PostgreSQL - создаём пользователя и базу данных

# Загружаем переменные из .env файла
load_dotenv()

# Получаем параметры из .env
DB_USER = os.getenv('POSTGRES_USER', 'maxim')
DB_PASSWORD = os.getenv('POSTGRES_PASSWORD', '12345678')
DB_NAME = os.getenv('POSTGRES_DB', 'tusur_db')

# 1. Запускаем PostgreSQL
subprocess.run(['sudo', 'service', 'postgresql', 'start'], check=True)

# Создаём пользователя и базу данных
create_user_cmd = f"sudo -u postgres psql -c \"CREATE USER {DB_USER} WITH PASSWORD '{DB_PASSWORD}';\""
create_db_cmd = f"sudo -u postgres psql -c \"CREATE DATABASE {DB_NAME} OWNER {DB_USER};\""
grant_superuser_cmd = f"sudo -u postgres psql -c \"ALTER USER {DB_USER} WITH SUPERUSER;\""

# Обернем вызовы subprocess.run в try-except, чтобы избежать остановки при уже существующем пользователе/БД
try:
    subprocess.run(create_user_cmd, shell=True, check=True, capture_output=True)
    print(f"Пользователь {DB_USER} создан.")
except subprocess.CalledProcessError as e:
    stderr_output = e.stderr.decode().strip() if e.stderr else "<нет вывода ошибки>"
    print(f"Предупреждение: Пользователь {DB_USER} уже существует или ошибка при создании: {stderr_output}")

try:
    subprocess.run(create_db_cmd, shell=True, check=True, capture_output=True)
    print(f"База данных {DB_NAME} создана.")
except subprocess.CalledProcessError as e:
    stderr_output = e.stderr.decode().strip() if e.stderr else "<нет вывода ошибки>"
    print(f"Предупреждение: База данных {DB_NAME} уже существует или ошибка при создании: {stderr_output}")

try:
    subprocess.run(grant_superuser_cmd, shell=True, check=True, capture_output=True)
    print(f"Права суперпользователя предоставлены {DB_USER}.")
except subprocess.CalledProcessError as e:
    stderr_output = e.stderr.decode().strip() if e.stderr else "<нет вывода ошибки>"
    print(f"Предупреждение: Ошибка при предоставлении прав суперпользователя: {stderr_output}")

Пользователь analyst создан.
База данных bpmn_models создана.
Права суперпользователя предоставлены analyst.


In [ ]:
# @title PostgreSQL - создание таблиц

# Инициализация PostgreSQL (асинхронный клиент библиотеки databases)
database = Database(POSTGRES_CONNECTION_ASYNC_URL)
metadata = MetaData()

# Определение структуры таблиц (перенесено из dependencies.py без изменений)
xml_documents_table = Table(
    "xml_documents",
    metadata,
    Column("id", PG_UUID(as_uuid=True), primary_key=True, default=uuid.uuid4),
    Column("parent_doc_id", String, index=True, nullable=False),
    Column("filename", String, nullable=False),
    Column("xml_content", Text, nullable=False),
    Column("upload_timestamp", DateTime, default=lambda: datetime.now(timezone.utc).replace(tzinfo=None)),
)

generated_results_table = Table(
    "generated_results",
    metadata,
    Column("id", PG_UUID(as_uuid=True), primary_key=True, default=uuid.uuid4),
    Column("parent_doc_id", String, index=True),
    Column("result_type", String),
    Column("full_prompt", Text, nullable=True), # <-- ДОБАВЛЕНО: Хранилище для полного промпта целиком
    Column("content", Text),
    Column("created_at", DateTime, default=lambda: datetime.now(timezone.utc).replace(tzinfo=None)),
)

logs_table = Table(
    "logs",
    metadata,
    Column("id", Integer, primary_key=True, autoincrement=True),
    Column("timestamp", DateTime, server_default=func.now(), index=True),
    Column("level", String),
    Column("event", String),
    Column("duration", Float),
    Column("context_size", Integer),
    Column("message", Text),
    Column("traceback", Text),
)

# Асинхронное подключение и создание таблиц
# В блокнотах Jupyter/Colab ключевое слово `await` можно использовать прямо на верхнем уровне
await database.connect()

engine = create_engine(POSTGRES_CONNECTION_SYNC_URL)
metadata.create_all(engine)

# Проверяем таблицы
inspector = inspect(engine)
tables = inspector.get_table_names()

print("Таблицы в базе данных:")
for table in tables:
    print(f"  - {table}")

# Дополнительно: смотрим структуру конкретной таблицы
if tables:
    print(f"\nСтруктура таблицы '{tables[0]}':")
    columns = inspector.get_columns(tables[0])
    for col in columns:
        print(f"  {col['name']} ({col['type']})")

print("✓ Подключение к PostgreSQL установлено. Таблицы проверены/созданы.")

Таблицы в базе данных:
  - xml_documents
  - generated_results
  - logs

Структура таблицы 'xml_documents':
  id (UUID)
  parent_doc_id (VARCHAR)
  filename (VARCHAR)
  xml_content (TEXT)
  upload_timestamp (TIMESTAMP)
✓ Подключение к PostgreSQL установлено. Таблицы проверены/созданы.


In [ ]:
# @title Таблицы - содержание

# Словарь для хранения датафреймов
dataframes = {}

# Дополнительно: смотрим структуру и выгружаем данные из каждой таблицы
for table_name in tables:
    print(f"\n{'='*50}")
    print(f"Обработка таблицы: {table_name}")

    # Смотрим структуру таблицы
    print(f"\nСтруктура таблицы '{table_name}':")
    columns = inspector.get_columns(table_name)
    for col in columns:
        print(f"  {col['name']} ({col['type']})")

    # Выгружаем содержимое таблицы в датафрейм
    try:
        df = pd.read_sql_table(table_name, engine)
        dataframes[table_name] = df

        # Выводим базовую информацию о датафрейме
        print(f"\nИнформация о данных:")
        print(f"  Количество строк: {len(df)}")
        print(f"  Количество колонок: {len(df.columns)}")

        if len(df) > 0:
            print(f"  Первые 3 строки:")
            display(df.head(3))  # Красиво отображается в Colab
        else:
            print("  Таблица пуста")

    except Exception as e:
        print(f"  Ошибка при загрузке данных: {e}")
        dataframes[table_name] = None

print(f"\n{'='*60}")
print("ВЫГРУЗКА ЗАВЕРШЕНА")
print(f"Всего обработано таблиц: {len(tables)}")
print(f"Успешно загружено в датафреймы: {sum(1 for df in dataframes.values() if df is not None)}")


Обработка таблицы: xml_documents

Структура таблицы 'xml_documents':
  id (UUID)
  parent_doc_id (VARCHAR)
  filename (VARCHAR)
  xml_content (TEXT)
  upload_timestamp (TIMESTAMP)

Информация о данных:
  Количество строк: 10
  Количество колонок: 5
  Первые 3 строки:


,id,parent_doc_id,filename,xml_content,upload_timestamp
0,0871b007-8335-43e5-a3db-9ff033d7bbee,a7cea93f5cdd0127e356c2a7f9f3eaa2fd1cffca866e48...,МФиНП.УМБР.ОМиП.02_Ведение пречня КБК.drawio,"<BPMN_Logical_Structure><FlowElement id=""umNi5...",NaT
1,a74d74aa-9cda-4a70-a39a-4975f0a40de1,4f25e9a910300f27262d943cdba9c996171dcdddb9168e...,Мониторинг качества финансового менеджмента.dr...,"<BPMN_Logical_Structure><FlowElement id=""CZMDA...",NaT
2,07665c77-e9c7-4554-8824-078ab5577fcf,0920bdfc897b10b59f4994ad56ffd96c66f105e5d7fcbb...,ведения кассового плана по доходам.drawio,"<BPMN_Logical_Structure><FlowElement id=""SXaTu...",NaT



Обработка таблицы: generated_results

Структура таблицы 'generated_results':
  id (UUID)
  parent_doc_id (VARCHAR)
  result_type (VARCHAR)
  full_prompt (TEXT)
  content (TEXT)
  created_at (TIMESTAMP)

Информация о данных:
  Количество строк: 0
  Количество колонок: 6
  Таблица пуста

Обработка таблицы: logs

Структура таблицы 'logs':
  id (INTEGER)
  timestamp (TIMESTAMP)
  level (VARCHAR)
  event (VARCHAR)
  duration (DOUBLE PRECISION)
  context_size (INTEGER)
  message (TEXT)
  traceback (TEXT)

Информация о данных:
  Количество строк: 10
  Количество колонок: 8
  Первые 3 строки:


,id,timestamp,level,event,duration,context_size,message,traceback
0,1,2026-06-12 05:10:14.576453,INFO,document_indexing,46.80,19319,Успешная индексация файла Приказ МФ и НП N 84-...,None
1,2,2026-06-12 05:10:16.172574,INFO,document_indexing,1.59,32471,Успешная индексация файла Регламент КФМ от 13....,None
2,3,2026-06-12 05:10:19.617864,INFO,document_indexing,3.44,107538,Успешная индексация файла Приказ МФ и НП Новос...,None



ВЫГРУЗКА ЗАВЕРШЕНА
Всего обработано таблиц: 3
Успешно загружено в датафреймы: 3


# ПРОЕКТ

## Ollama

In [ ]:
# @title Ollama - устанавливаем сервер

print("Устанавливаем зависимости для обнаружения GPU...")
!sudo apt-get update
!sudo apt-get install -y zstd lspci lshw

print("Устанавливаем zstd, необходимый для Ollama...")
!sudo apt-get install -y zstd

print("Продолжаем установку Ollama...")
!curl -fsSL https://ollama.com/install.sh -o install_ollama.sh
!chmod +x install_ollama.sh
!./install_ollama.sh

print("Если установка прошла успешно, Ollama должна быть готова.")

Устанавливаем зависимости для обнаружения GPU...
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://apt.postgresql.org/pub/repos/apt jammy-pgdg InRelease
Hit:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r

In [ ]:
# @title Ollama - фоновый запуск

print("Запускаем Ollama сервер в фоновом режиме...")

# Настройка окружения
ollama_env = os.environ.copy()
ollama_env["OLLAMA_HOST"] = "127.0.0.1:11434"
ollama_env["OLLAMA_CUDA"] = "1"  # <-- ДОБАВЛЕНА СТРОКА: активируем использование GPU

# Путь к файлу, куда будут записываться логи сервера
LOG_FILE_PATH = "/content/ollama_server.log"

# Открываем файл на запись. 'w' очищает старые логи при каждом перезапуске ячейки
ollama_log_file = open(LOG_FILE_PATH, "w", encoding="utf-8")

# 1. Запускаем фоновый процесс сервера, перенаправляя потоки вывода в файл
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=ollama_log_file,
    stderr=ollama_log_file,
    env=ollama_env,
    cwd="/content",
)

# 2. Ждем готовности сервера по HTTP
ollama_url = "http://127.0.0.1:11434"
for i in range(30):  # Проверяем в течение 60 секунд (30 итераций по 2 секунды)
    try:
        if requests.get(ollama_url, timeout=1).status_code == 200:
            print(f"✅ Ollama успешно запущена и готова к работе! (заняло {i*2} сек.)")
            print(f"📝 Логи сервера пишутся в файл: {LOG_FILE_PATH}")
            break
    except (requests.exceptions.ConnectionError, requests.exceptions.Timeout):
        pass
    time.sleep(2)
else:
    print("❌ Ошибка: Сервер Ollama не ответил за 60 секунд.")
    print("🛑 Выводим содержимое файла логов для дебага:\n")

    # Закрываем дескриптор файла перед чтением, чтобы сбросить буфер на диск
    ollama_log_file.close()

    # Читаем логи из файла и печатаем их на экран
    if os.path.exists(LOG_FILE_PATH):
        with open(LOG_FILE_PATH, "r", encoding="utf-8") as f:
            print(f.read())
    else:
        print("Файл логов не был создан.")


Запускаем Ollama сервер в фоновом режиме...
✅ Ollama успешно запущена и готова к работе! (заняло 4 сек.)
📝 Логи сервера пишутся в файл: /content/ollama_server.log


In [ ]:
# @title Ollama - установка моделей

print("Скачивание моделей в Google Colab...")
subprocess.run(["ollama", "pull", "qwen2.5:14b-instruct-q4_K_M"], check=True, env=ollama_env)
subprocess.run(["ollama", "pull", "snowflake-arctic-embed2:latest"], check=True, env=ollama_env)
print("Службы полностью готовы к работе!")

Скачивание моделей в Google Colab...
Службы полностью готовы к работе!


In [ ]:
print("Проверка доступности сервера Ollama...")
ollama_url = "http://127.0.0.1:11434"

try:
    response = requests.get(ollama_url, timeout=5)
    if response.status_code == 200:
        print(f"✅ Сервер Ollama доступен по адресу: {ollama_url}")

        print("\nПроверка установленных моделей Ollama...")
        # ollama_env должен быть определен в предыдущих ячейках
        if 'ollama_env' in locals() or 'ollama_env' in globals():
            list_models_process = subprocess.run(
                ['ollama', 'list'],
                capture_output=True,
                text=True,
                check=False,
                env=ollama_env
            )

            if list_models_process.returncode == 0:
                print(list_models_process.stdout)
            else:
                print(f"❌ Ошибка при получении списка моделей: {list_models_process.stderr}")
        else:
            print("❌ Переменная 'ollama_env' не найдена. Убедитесь, что ячейка с запуском Ollama выполнена.")

    else:
        print(f"❌ Сервер Ollama не вернул успешный статус (HTTP {response.status_code})")
except requests.exceptions.ConnectionError:
    print("❌ Сервер Ollama недоступен. Убедитесь, что он запущен.")
except requests.exceptions.Timeout:
    print("❌ Тайм-аут подключения к серверу Ollama.")
except Exception as e:
    print(f"❌ Произошла непредвиденная ошибка: {e}")


Проверка доступности сервера Ollama...
✅ Сервер Ollama доступен по адресу: http://127.0.0.1:11434

Проверка установленных моделей Ollama...
NAME                              ID              SIZE      MODIFIED       
snowflake-arctic-embed2:latest    5de93a84837d    1.2 GB    3 seconds ago     
qwen2.5:14b-instruct-q4_K_M       7cdf5a0187d5    9.0 GB    18 seconds ago    



In [ ]:
# @title Ollama - инициализация

# ПРАВИЛЬНАЯ КЛАССИЧЕСКАЯ ИНИЦИАЛИЗАЦИЯ
llm = ChatOllama(
    model=LLM_MODEL,
    temperature=0.25,
    num_ctx=32000,
    base_url="http://127.0.0.1:11434",
    model_kwargs={
        "stop": ["</bpmn:definitions>", "</bpmn2:definitions>", "</definitions>", "</BPMN_Logical_Structure>"]
    }
)


In [ ]:
from langchain_core.messages import HumanMessage

# Отправляем тестовый запрос к модели
print("Отправка тестового запроса к модели qwen2.5:14b-instruct-q4_K_M...")
response = await llm.ainvoke([HumanMessage(content="Привет, как тебя зовут?")])

print("Ответ модели:")
print(response.content)


## RAG - компоненты системы

In [ ]:
print("Начало инициализации компонентов...")

# 1. Инициализация модели эмбеддингов
embedding_model = OllamaEmbeddings(
    model=EMBEDDINGS_MODEL,
    base_url="http://127.0.0.1:11434",
    num_ctx=8192
  )

# 2. Подключение к векторному хранилищу Chroma
chroma = AsyncChroma(
    collection_name=CHROMA_COLLECTION,
    persist_directory=CHROMA_PATH,
    embedding_function=embedding_model,
)

# 3. Инициализация менеджера записей (SQLite) для дедупликации чанков
record_manager = SQLRecordManager(
    namespace="rag_bpmn",
    db_url=RECORD_MANAGER,
    async_mode=True  # Асинхронный режим поддерживается в Colab "из коробки"
)
# Асинхронно создаем таблицы и схему данных в SQLite, если они еще не созданы
await record_manager.acreate_schema()

print("""Создали три готовых компонента для системы RAG:

embedding_model — модель для преобразования текста в векторы;

chroma — векторная база данных для хранения и поиска похожих фрагментов;

record_manager — менеджер для предотвращения дублирования данных.

Эти компоненты обычно используются вместе для построения системы, которая:

загружает документы;
разбивает их на чанки;
удаляет дубликаты с помощью record_manager;
сохраняет уникальные чанки в chroma с эмбеддингами от embedding_model;
выполняет семантический поиск по запросам пользователя.
""")


Начало инициализации компонентов...
Создали три готовых компонента для системы RAG:

embedding_model — модель для преобразования текста в векторы;

chroma — векторная база данных для хранения и поиска похожих фрагментов;

record_manager — менеджер для предотвращения дублирования данных.

Эти компоненты обычно используются вместе для построения системы, которая:

загружает документы;
разбивает их на чанки;
удаляет дубликаты с помощью record_manager;
сохраняет уникальные чанки в chroma с эмбеддингами от embedding_model;
выполняет семантический поиск по запросам пользователя.



### Просмотр содержимого ChromaDB (через SQLite)

In [ ]:
# Путь к файлу базы данных ChromaDB
chroma_db_path = f"{CHROMA_PATH}/chroma.sqlite3"

if not os.path.exists(chroma_db_path):
    print(f"Ошибка: Файл базы данных ChromaDB не найден по пути: {chroma_db_path}")
else:
    print(f"Подключаемся к базе данных ChromaDB: {chroma_db_path}")
    try:
        # Подключение к SQLite базе данных
        conn = sqlite3.connect(chroma_db_path)
        cursor = conn.cursor()

        # Получение списка таблиц
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
        tables_sql = cursor.fetchall()
        print("\nТаблицы в chroma.sqlite3:")
        for table in tables_sql:
            print(f"  - {table[0]}")

        # Попытка прочитать основную таблицу (обычно 'embeddings' или 'items')
        # Название таблицы может варьироваться в зависимости от версии ChromaDB
        # и конфигурации, но часто это 'embeddings' или 'items'
        # Проверим, есть ли таблица 'embeddings' или 'items'
        main_table_found = False
        for table_name_tuple in tables_sql:
            table_name = table_name_tuple[0]
            if 'embeddings' in table_name.lower() or 'items' in table_name.lower() or 'documents' in table_name.lower(): # Более гибкий поиск
                try:
                    print(f"\nПервые 5 строк таблицы '{table_name}':")
                    df_chroma = pd.read_sql(f"SELECT * FROM {table_name} LIMIT 5", conn)
                    display(df_chroma)
                    main_table_found = True
                    # Если найдена подходящая таблица, можно выйти из цикла
                    break
                except pd.io.sql.DatabaseError as e:
                    print(f"Не удалось прочитать таблицу {table_name}: {e}")

        if not main_table_found:
            print("\nНе найдена основная таблица с данными (embeddings/items/documents) для отображения.\nВозможно, база данных пуста или структура отличается.")

        # Закрытие соединения
        conn.close()
        print("\nСоединение с ChromaDB закрыто.")

    except sqlite3.Error as e:
        print(f"Ошибка при доступе к ChromaDB: {e}")


Подключаемся к базе данных ChromaDB: /content/Project/data/vector_store/chroma.sqlite3

Таблицы в chroma.sqlite3:
  - migrations
  - acquire_write
  - collection_metadata
  - segment_metadata
  - tenants
  - databases
  - collections
  - maintenance_log
  - segments
  - embeddings
  - embedding_metadata
  - max_seq_id
  - embedding_fulltext_search
  - embedding_fulltext_search_data
  - embedding_fulltext_search_idx
  - embedding_fulltext_search_content
  - embedding_fulltext_search_docsize
  - embedding_fulltext_search_config
  - embedding_metadata_array
  - embeddings_queue
  - embeddings_queue_config

Первые 5 строк таблицы 'embeddings':


,id,segment_id,embedding_id,seq_id,created_at
0,1,fa52e829-9708-4f73-b770-aee52a158286,3c387534-17f8-5e10-ac99-58dc5877bdd5,1,2026-06-12 05:10:14
1,2,fa52e829-9708-4f73-b770-aee52a158286,822a7bac-bda2-57e2-aedc-24ccd7c7fbbd,2,2026-06-12 05:10:14
2,3,fa52e829-9708-4f73-b770-aee52a158286,c8796a70-2ff8-5fac-bac0-8178e6078d79,3,2026-06-12 05:10:14
3,4,fa52e829-9708-4f73-b770-aee52a158286,1dad37f3-8cf2-5d49-83fb-80f4d02815d2,4,2026-06-12 05:10:14
4,5,fa52e829-9708-4f73-b770-aee52a158286,1c4f48ce-a029-588a-95fe-4a5d2c3bf5d6,5,2026-06-12 05:10:14



Соединение с ChromaDB закрыто.


## ФАЙЛ - загрузка

### Вспомогательные функции

In [ ]:
# @title Очистка XML

def clean_drawio_xml(raw_xml_text: str) -> str:
    """
    Очищает XML-код draw.io/mxfile от графического мусора.
    Максимально глубоко очищает Base64 строки и подбирает 3 режима zlib
    для гарантированного обхода ошибки Error -5.
    """
    if not raw_xml_text or not raw_xml_text.strip():
        return ""

    try:
        cleaned_source = re.sub(r"```xml\s*|```", "", raw_xml_text).strip()
        root = ET.fromstring(cleaned_source)

        # Ищем тег <diagram>
        diagram_node = root.find(".//diagram") if root.tag != "diagram" else root

        if diagram_node is not None and diagram_node.text:
            # ОЧИСТКА СТРОКИ: Удаляем любые переносы строк, пробелы и табы внутри Base64
            encoded_text = "".join(diagram_node.text.split()).strip()

            # Строгая проверка: если строка содержит XML-теги, это НЕ сжатый Base64!
            if "<" not in encoded_text and len(encoded_text) > 20:
                is_decompressed = False
                decompressed_bytes = None

                # Пробуем декодировать Base64
                try:
                    raw_bytes = base64.b64decode(encoded_text)
                except Exception as b64_err:
                    print(f"⚠️ Ошибка Base64-декодирования: {b64_err}")
                    raw_bytes = None

                if raw_bytes:
                    # ПОПЫТКА 1: Сырой Deflate без заголовков (wbits=-15) - стандарт Draw.io
                    try:
                        decompressed_bytes = zlib.decompress(raw_bytes, -15)
                        is_decompressed = True
                    except zlib.error:
                        pass

                    # ПОПЫТКА 2: Если Попытка 1 выдала Error -5, пробуем gzip/zlib автоопределение
                    if not is_decompressed:
                        try:
                            decompressed_bytes = zlib.decompress(raw_bytes, zlib.MAX_WBITS | 16)
                            is_decompressed = True
                        except zlib.error:
                            pass

                    # ПОПЫТКА 3: Стандартный декомпрессор со стандартным окном (wbits=0/авто)
                    if not is_decompressed:
                        try:
                            decompressed_bytes = zlib.decompress(raw_bytes)
                            is_decompressed = True
                        except zlib.error:
                            pass

                # Если один из трех режимов сработал — собираем XML дерево
                if is_decompressed and decompressed_bytes:
                    try:
                        decoded_string = urllib.parse.unquote(decompressed_bytes.decode('utf-8'))
                        root = ET.fromstring(decoded_string)
                        print("✨ Сжатый файл .drawio успешно декомпрессирован.")
                    except Exception as parse_inner_err:
                        print(f"⚠️ Ошибка парсинга распакованного XML: {parse_inner_err}")
                else:
                    # Абсолютный предохранитель: если это не распаковалось, но и тегов нет —
                    # выводим сообщение, но не ломаем выполнение.
                    print(f"ℹ️ Содержимое тега diagram не требует распаковки или имеет кастомный формат.")

        # Стандартный блок извлечения сущностей (кубиков и стрелок)
        clean_root = ET.Element("BPMN_Logical_Structure")

        for cell in root.iter("mxCell"):
            cell_id = cell.get("id")
            value = cell.get("value")
            source = cell.get("source")
            target = cell.get("target")
            edge = cell.get("edge")

            if value:
                value = re.sub(r"<[^>]+>", "", value).strip()

            if edge == "1" and source and target:
                edge_el = ET.SubElement(clean_root, "SequenceFlow")
                edge_el.set("id", cell_id)
                if value: edge_el.set("name", value)
                edge_el.set("sourceRef", source)
                edge_el.set("targetRef", target)
            elif value and not edge:
                node_el = ET.SubElement(clean_root, "FlowElement")
                node_el.set("id", cell_id)
                node_el.set("name", value)

        return ET.tostring(clean_root, encoding="utf-8").decode("utf-8")

    except Exception as global_err:
        print(f"ℹ️ Обработка файла как сырого BPMN (без изменений): {global_err}")
        return raw_xml_text

### Функции загрузки, чтения и индексации документов (Chroma + RecordManager)

In [ ]:
# Настройка правил разбиения на чанки
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=150,
    length_function=len,
    is_separator_regex=False,
    separators=["\n", "\n\n", "."],
)

def _read_local_document(file_path: str) -> str:
    """Читает текст из файлов (.txt, .pdf, .docx) прямо с диска."""
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Файл не найден: {file_path}")

    file_extension = os.path.splitext(file_path)[1].lower()
    content = ""

    if file_extension == ".txt":
        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read()
    elif file_extension == ".pdf":
        loader = PyPDFLoader(file_path)
        pages = loader.load()
        content = "\n".join([p.page_content for p in pages])
    elif file_extension == ".docx":
        doc = docx.Document(file_path)
        content = "\n".join([para.text for para in doc.paragraphs])
    else:
        raise ValueError(f"Неподдерживаемый формат: {file_extension}")
    return content

async def upload_and_index_pipeline(doc_path: str, xml_path: str):
    """
    Полный цикл: чтение, генерация детерминированного parent_doc_id,
    запись XML в Postgres, умная индексация чанков текста в ChromaDB
    и автоматическое логирование всех этапов в PostgreSQL logs_table.
    """
    start_time = time.time()
    print(f"=== Старт обработки файлов ===")

    # Инициализируем переменные базовыми значениями на случай ранней ошибки
    context_size = 0
    doc_name = os.path.basename(doc_path) if os.path.exists(doc_path) else "unknown"

    try:
        # 1. Читаем основной документ и создаем его хэш-ID
        main_content = _read_local_document(doc_path)
        context_size = len(main_content)  # Считаем размер контекста в символах

        if not main_content.strip():
            raise ValueError("Текст основного документа пуст!")

        parent_doc_id = hashlib.sha256(main_content.encode("utf-8")).hexdigest()
        xml_name = os.path.basename(xml_path)

        print(f"📝 Документ: {doc_name}")
        print(f"🔑 Сгенерирован детерминированный ID: {parent_doc_id}")

        # 2. Читаем сопутствующий XML-файл
        if not os.path.exists(xml_path):
            raise FileNotFoundError(f"XML файл не найден по пути {xml_path}")

        with open(xml_path, "r", encoding="utf-8") as f:
            xml_content = f.read()

        if not xml_content.strip():
            raise ValueError("Содержимое XML файла пусто!")

        # 2.1. Очистка XML от графических тегов
        xml_content = clean_drawio_xml(xml_content)

        # 3. Запись метаданных XML в PostgreSQL
        db_query = xml_documents_table.insert().values(
            id=uuid.uuid4(),
            parent_doc_id=parent_doc_id,
            filename=xml_name,
            xml_content=xml_content
        )
        await database.execute(db_query)
        print(f"💾 [Postgres] XML-схема успешно сохранена.")

        # 4. Нарезка текста на чанки с метаданными привязки
        metadata_template = {
            "parent_doc_id": parent_doc_id,
            "original_filename": doc_name,
            "xml_filename": xml_name
        }

        chunks = text_splitter.create_documents(
            texts=[main_content],
            metadatas=[metadata_template]
        )
        print(f"✂️  Текст успешно разбит на {len(chunks)} чанков.")

        # 5. Инкрементальная индексация через LangChain aindex
        print(f"⏳ Синхронизация с ChromaDB через SQLRecordManager...")
        index_result = await aindex(
            chunks,
            record_manager=record_manager,
            vector_store=chroma,
            cleanup="incremental",
            source_id_key="parent_doc_id"
        )

        duration = round(time.time() - start_time, 2)
        print(f"✅ [ChromaDB] Синхронизация завершена за {duration} сек.")
        print(f"📊 Статус индексации: {index_result}")

        # --- ДОБАВЛЕНО LOG_INFO В POSTGRES ---
        log_query = logs_table.insert().values(
            level="INFO",
            event="document_indexing",
            duration=duration,
            context_size=context_size,
            message=f"Успешная индексация файла {doc_name}. Создано чанков: {len(chunks)}. Результат менеджера: {index_result}",
            traceback=None
        )
        await database.execute(log_query)

        return parent_doc_id

    except Exception as e:
        duration = round(time.time() - start_time, 2)
        error_msg = str(e)
        error_traceback = traceback.format_exc()

        print(f"💥 Критическая ошибка в конвейере: {error_msg}")

        # --- ДОБАВЛЕНО LOG_ERROR В POSTGRES ---
        try:
            log_query = logs_table.insert().values(
                level="ERROR",
                event="db_error",
                duration=duration,
                context_size=context_size,
                message=f"Ошибка при обработке файла {doc_name}: {error_msg}",
                traceback=error_traceback
            )
            await database.execute(log_query)
            print("🛑 Ошибка успешно залогирована в таблицу logs_table.")
        except Exception as log_exception:
            print(f"⚠️ Не удалось записать лог ошибки в базу: {log_exception}")

        return None

print("✓ Логика загрузки и индексации документов успешно инициализирована.")

✓ Логика загрузки и индексации документов успешно инициализирована.


In [ ]:
async def process_all_folders(root_folder_path: str, upload_function, log_file_path: str = None):
    """
    Обходит все папки в указанной директории, находит пары файлов
    (текстовый файл + XML/drawio), конвертирует пути в строки и передаёт
    их в асинхронную функцию загрузки в БД.

    Пишет лог в текстовый файл и выводит статус на экран.
    """
    start_process_time = time.time()
    root_path = Path(root_folder_path)

    if not root_path.exists():
        print(f"❌ Ошибка: Корневая папка не найдена: {root_folder_path}")
        return

    # Настройка пути к файлу логов, если он не передан
    if log_file_path is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        log_file_path = f"folder_processor_{timestamp}.log"

    log_file = Path(log_file_path)

    # Вспомогательная функция для одновременной записи в файл и на экран
    def log_message(message: str, level: str = "INFO"):
        time_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        full_line = f"[{time_str}] [{level}] {message}"
        # Выводим в Colab
        print(full_line if level != "❌ Ошибка" else f"❌ {full_line}")
        # Пишем в файл на Диск
        with open(log_file, "a", encoding="utf-8") as f:
            f.write(full_line + "\n")

    log_message(f"Запуск обработки папок. Корневая папка: {root_folder_path}")
    log_message(f"Файл логов создается по пути: {log_file.resolve()}")
    log_message("=" * 60)

    total_folders = 0
    processed_pairs = 0
    skipped_folders = 0

    for folder_path in root_path.iterdir():
        if folder_path.is_dir():
            total_folders += 1
            log_message(f"Обрабатываем папку [{total_folders}]: {folder_path.name}")

            text_files = []
            xml_files = []

            for file_path in folder_path.iterdir():
                if file_path.is_file():
                    ext = file_path.suffix.lower()
                    if ext in ['.txt', '.pdf', '.docx']:
                        text_files.append(file_path)
                    elif ext in ['.xml', '.drawio']:
                        xml_files.append(file_path)

            if len(text_files) == 1 and len(xml_files) == 1:
                # ИСПРАВЛЕНО: Явное извлечение первого элемента [0] и перевод в str
                text_str_path = str(text_files[0])
                xml_str_path = str(xml_files[0])

                log_message(f"   ↳ Найдена пара: {text_files[0].name} + {xml_files[0].name}")

                try:
                    await upload_function(text_str_path, xml_str_path)
                    log_message(f"   ✓ Успешно загружено в БД.")
                    processed_pairs += 1
                except Exception as e:
                    log_message(f"Ошибка при загрузке файлов из папки {folder_path.name}: {e}", level="ERROR")
            else:
                skipped_folders += 1
                log_message(f"Пропущено (некорректный набор файлов):", level="WARNING")
                log_message(f"   Текстовые: {[f.name for f in text_files]}", level="WARNING")
                log_message(f"   XML/drawio: {[f.name for f in xml_files]}", level="WARNING")
            log_message("-" * 50)

    duration = round(time.time() - start_process_time, 2)
    success_rate = (processed_pairs / total_folders * 100) if total_folders > 0 else 0.0

    log_message("=" * 60)
    log_message("ИТОГОВАЯ СТАТИСТИКА ПАКЕТНОЙ ЗАГРУЗКИ:")
    log_message(f"⏱️  Общее время выполнения: {duration} сек.")
    log_message(f"📂 Всего обнаружено папок: {total_folders}")
    log_message(f"✅ Успешно обработано пар: {processed_pairs}")
    log_message(f"❌ Пропущено папок: {skipped_folders}")
    log_message(f"📈 Процент успешного выполнения: {success_rate:.1f}%")
    log_message("=" * 60)
    log_message("Обработка завершена.")

### Загрузка

In [ ]:
# Укажите путь к вашей корневой папке на Google Диске, где лежат папки с документами
ROOT_DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/RAG_BPMN/reference_documents"

# Передаем путь и нашу функцию из предыдущего шага в качестве аргумента
await process_all_folders(
    root_folder_path=ROOT_DATA_DIR,
    upload_function=upload_and_index_pipeline
)


[2026-06-12 05:09:27] [INFO] Запуск обработки папок. Корневая папка: /content/Project/data/reference_documents
[2026-06-12 05:09:27] [INFO] Файл логов создается по пути: /content/Project/folder_processor_20260612_050927.log
[2026-06-12 05:09:27] [INFO] ============================================================
[2026-06-12 05:09:27] [INFO] Обрабатываем папку [1]: МФиНП.УМБР.ОМиП.02_Ведение пречня КБК_СОГЛАСОВАНО
[2026-06-12 05:09:27] [INFO]    ↳ Найдена пара: Приказ МФ и НП N 84-НПА.pdf + МФиНП.УМБР.ОМиП.02_Ведение пречня КБК.drawio
=== Старт обработки файлов ===
📝 Документ: Приказ МФ и НП N 84-НПА.pdf
🔑 Сгенерирован детерминированный ID: a7cea93f5cdd0127e356c2a7f9f3eaa2fd1cffca866e485ce779e99a668b0b52
💾 [Postgres] XML-схема успешно сохранена.
✂️  Текст успешно разбит на 15 чанков.
⏳ Синхронизация с ChromaDB через SQLRecordManager...


/usr/local/lib/python3.12/dist-packages/langchain_core/indexing/api.py:748: UserWarning: Using SHA-1 for document hashing. SHA-1 is *not* collision-resistant; a motivated attacker can construct distinct inputs that map to the same fingerprint. If this matters in your threat model, switch to a stronger algorithm such as 'blake2b', 'sha256', or 'sha512' by specifying  `key_encoder` parameter in the `index` or `aindex` function. 
  _warn_about_sha1()


✅ [ChromaDB] Синхронизация завершена за 46.8 сек.
📊 Статус индексации: {'num_added': 14, 'num_updated': 0, 'num_skipped': 1, 'num_deleted': 0}
[2026-06-12 05:10:14] [INFO]    ✓ Успешно загружено в БД.
[2026-06-12 05:10:14] [INFO] --------------------------------------------------
[2026-06-12 05:10:14] [INFO] Обрабатываем папку [2]: Мониторинг качества финансового менеджмента
[2026-06-12 05:10:14] [INFO]    ↳ Найдена пара: Регламент КФМ от 13.04.2023.pdf.txt + Мониторинг качества финансового менеджмента.drawio
=== Старт обработки файлов ===
📝 Документ: Регламент КФМ от 13.04.2023.pdf.txt
🔑 Сгенерирован детерминированный ID: 4f25e9a910300f27262d943cdba9c996171dcdddb9168ebb9ebbc3a455a4eef9
💾 [Postgres] XML-схема успешно сохранена.
✂️  Текст успешно разбит на 24 чанков.
⏳ Синхронизация с ChromaDB через SQLRecordManager...
✅ [ChromaDB] Синхронизация завершена за 1.59 сек.
📊 Статус индексации: {'num_added': 24, 'num_updated': 0, 'num_skipped': 0, 'num_deleted': 0}
[2026-06-12 05:10:16] [INFO

In [ ]:
# @title Просмотрщик системных логов

import pandas as pd
from IPython.display import display, HTML

async def show_latest_logs(limit: int = 10):
    """
    Запрашивает последние логи из PostgreSQL и выводит их в виде интерактивной таблицы.
    """
    # 1. Формируем SQL-запрос на выборку последних записей по времени
    query = logs_table.select().order_by(logs_table.c.timestamp.desc()).limit(limit)
    rows = await database.fetch_all(query)

    if not rows:
        print("📭 Таблица логов пока пуста.")
        return

    # 2. Переводим результаты в список словарей для pandas
    logs_list = [dict(row) for row in rows]
    df = pd.DataFrame(logs_list)

    # Косметика: округлим длительность операции для читаемости
    if 'duration' in df.columns:
        df['duration'] = df['duration'].round(2)

    print(f"📊 Отображены последние {len(df)} записей из таблицы логов:")

    # 3. Настройка красивого вывода в Colab (подсветка ошибок красным)
    def style_rows(row):
        if row['level'] == 'ERROR':
            return ['background-color: #ffcccc'] * len(row)
        return [''] * len(row)

    # Выводим таблицу без столбца traceback (чтобы не загромождать экран)
    columns_to_show = [col for col in df.columns if col != 'traceback']
    styled_df = df[columns_to_show].style.apply(style_rows, axis=1)
    display(styled_df)

    # 4. Если есть ошибки, выведем их стек вызовов (traceback) отдельно внизу
    error_rows = df[df['level'] == 'ERROR']
    if not error_rows.empty:
        print("\n🔍 Детализация критических ошибок (Traceback):")
        for idx, err in error_rows.iterrows():
            print(f"\n--- Ошибка ID {err['id']} [{err['timestamp']}] ---")
            print(f"Событие: {err['event']} | Сообщение: {err['message']}")
            print(f"Стек вызова:\n{err['traceback'] if err['traceback'] else 'Нет данных'}")
            print("-" * 50)

# ==========================================
# ЗАПУСК ПРОСМОТРЩИКА
# ==========================================
# Вы можете менять число в скобках, чтобы увидеть больше или меньше записей
await show_latest_logs(limit=5)


📊 Отображены последние 5 записей из таблицы логов:


,id,timestamp,level,event,duration,context_size,message
0,16,2026-06-13 03:02:07.034756,ERROR,json_generation,717.450000,0,UnboundLocalError: cannot access local variable 'raw_json' where it is not associated with a value
1,15,2026-06-13 02:50:09.490065,INFO,xml_generation,1045.320000,40000,Успешная генерация валидного XML (Без чанков). ID: 65197e24-da49-424b-b93e-0c48289ac726
2,14,2026-06-13 02:13:00.763447,ERROR,json_validation_error,106.840000,40000,"Битый JSON от модели. Ошибка: Expecting ',' delimiter: line 21 column 4 (char 424)"
3,13,2026-06-13 02:11:13.920663,ERROR,xml_validation_error,279.420000,40000,"Битый XML от модели. Ошибка: unclosed token: line 14, column 22"
4,12,2026-06-13 02:03:15.683697,ERROR,json_generation,0.010000,0,NameError: name '_read_local_document' is not defined



🔍 Детализация критических ошибок (Traceback):

--- Ошибка ID 16 [2026-06-13 03:02:07.034756] ---
Событие: json_generation | Сообщение: UnboundLocalError: cannot access local variable 'raw_json' where it is not associated with a value
Стек вызова:
Traceback (most recent call last):
  File "/tmp/ipykernel_2498/3481654913.py", line 99, in generate_process_json
    start_bracket_match = re.search(r"\{|\[", raw_json)
                                              ^^^^^^^^
UnboundLocalError: cannot access local variable 'raw_json' where it is not associated with a value

--------------------------------------------------

--- Ошибка ID 14 [2026-06-13 02:13:00.763447] ---
Событие: json_validation_error | Сообщение: Битый JSON от модели. Ошибка: Expecting ',' delimiter: line 21 column 4 (char 424)
Стек вызова:
[
  {
    "id": "start_01",
    "type": "startEvent",
    "name": "Представление проекта правового акта"
  },
  {
    "id": "task_01",
    "type": "task",
    "role": "Разработчик проект

In [ ]:
# @title Просмотрщик таблиц PostgreSQL и SQLite

async def inspect_all_tables():
    """
    Запрашивает данные из всех таблиц PostgreSQL и внутренней таблицы
    SQLRecordManager (SQLite), выводя их в читаемом виде.
    """
    print("=== АНАЛИЗ СОДЕРЖИМОГО БАЗ ДАННЫХ ===\n")

    # ----------------------------------------------------
    # ЧАСТЬ 1: Просмотр таблиц в PostgreSQL
    # ----------------------------------------------------
    postgres_tables = {
        "Документы XML (xml_documents)": xml_documents_table,
        "Результаты генерации LLM (generated_results)": generated_results_table,
        "Системные логи (logs)": logs_table
    }

    for title, table_obj in postgres_tables.items():
        display(Markdown(f"### 💾 Таблица PostgreSQL: **{title}**"))

        # Берем последние 10 записей для демонстрации
        query = table_obj.select().limit(10)
        rows = await database.fetch_all(query)

        if rows:
            df_pg = pd.DataFrame([dict(r) for r in rows])
            # Ограничим вывод длинных текстов, чтобы таблица не разъезжалась
            for text_col in ['xml_content', 'content', 'message', 'traceback']:
                if text_col in df_pg.columns:
                    df_pg[text_col] = df_pg[text_col].astype(str).str.slice(0, 100) + "..."
            display(df_pg)
        else:
            print("📭 Таблица пуста.\n")

    # ----------------------------------------------------
    # ЧАСТЬ 2: Просмотр кэша SQLRecordManager (SQLite)
    # ----------------------------------------------------
    display(Markdown("### 🗃️ Таблица SQLRecordManager: **Кэш чанков (SQLite)**"))
    try:
        # Извлекаем чистый путь к файлу SQLite из строки подключения SQLAlchemy
        # Строка выглядит так: sqlite+aiosqlite:///путь_к_файлу
        sqlite_url = RECORD_MANAGER.replace("sqlite+aiosqlite:///", "sqlite:///")
        engine_sqlite = create_engine(sqlite_url)

        # По умолчанию LangChain создает таблицу с именем 'upsertion_record'
        query_sqlite = "SELECT * FROM upsertion_record LIMIT 10;"
        df_sqlite = pd.read_sql(query_sqlite, con=engine_sqlite)

        if not df_sqlite.empty:
            display(df_sqlite)

            # Выведем общую статистику по чанкам
            total_chunks = pd.read_sql("SELECT COUNT(*) as count FROM upsertion_record;", con=engine_sqlite).iloc[0]['count']
            print(f"📊 Всего уникальных записей (чанков) под управлением менеджера: {total_chunks}")
        else:
            print("📭 Менеджер записей пока не отслеживает ни одного чанка.\n")

    except Exception as e:
        print(f"⚠️ Не удалось прочитать таблицу SQLite (возможно, она еще не создана LangChain): {e}")

# ==========================================
# ЗАПУСК АНАЛИЗА
# ==========================================
await inspect_all_tables()


=== АНАЛИЗ СОДЕРЖИМОГО БАЗ ДАННЫХ ===



### 💾 Таблица PostgreSQL: **Документы XML (xml_documents)**

,id,parent_doc_id,filename,xml_content,upload_timestamp
0,0871b007-8335-43e5-a3db-9ff033d7bbee,a7cea93f5cdd0127e356c2a7f9f3eaa2fd1cffca866e48...,МФиНП.УМБР.ОМиП.02_Ведение пречня КБК.drawio,"<BPMN_Logical_Structure><FlowElement id=""umNi5...",None
1,a74d74aa-9cda-4a70-a39a-4975f0a40de1,4f25e9a910300f27262d943cdba9c996171dcdddb9168e...,Мониторинг качества финансового менеджмента.dr...,"<BPMN_Logical_Structure><FlowElement id=""CZMDA...",None
2,07665c77-e9c7-4554-8824-078ab5577fcf,0920bdfc897b10b59f4994ad56ffd96c66f105e5d7fcbb...,ведения кассового плана по доходам.drawio,"<BPMN_Logical_Structure><FlowElement id=""SXaTu...",None
3,de3a209c-d8c3-4105-9329-b46299ba03d8,ffb0c847038a44dead85e9340785e04326ed8ae3e15d27...,МФиНП.УМБР.ОСИМБиФС.05.drawio,"<BPMN_Logical_Structure><FlowElement id=""FiubD...",None
4,b37dcbc4-a8cd-4f3f-b772-1199f1871e0f,54dfae407865d6662a0a1b643892b6981da935c16f9886...,МФиНП.УМБР.ОСИМБиФС.04.drawio,"<BPMN_Logical_Structure><FlowElement id=""FiubD...",None
5,c22a3e67-9767-4229-a5d2-36b0d8923851,043fdbc8518c91bfb5ace1d68abbe6557a33caa7ade605...,Процесс про наказы.drawio,"<BPMN_Logical_Structure><FlowElement id=""-f9v_...",None
6,d110d76c-5a25-437e-a436-63686c80c8c5,1780ca9955b4732146a871e542973af705493a5b0e0334...,МФиНП.УМБР.ОСИМБиФС.03.drawio,"<BPMN_Logical_Structure><FlowElement id=""FiubD...",None
7,b63b1cee-e511-4381-aea8-d4333d78c327,45d2d709c6938b392295b60184b717106b9f8aeac1d185...,МФиНП.УМБР.ОМиП.01.drawio,"<BPMN_Logical_Structure><FlowElement id=""6w6FR...",None
8,25ef0d75-01d2-40c0-9454-d26b413d2b34,329ee8af054a4d7362605c4a6ddb47e0586d85ed5ac38d...,Экспертиза исполнительных документов версия 2....,"<BPMN_Logical_Structure><FlowElement id=""ckplR...",None
9,f91db889-4837-4feb-9ba0-f0b6ec17012d,0eedd5883d376dc1d31c72a60b4c903ad9c452269021e7...,коды доходов.drawio,"<BPMN_Logical_Structure><FlowElement id=""LmCZS...",None


### 💾 Таблица PostgreSQL: **Результаты генерации LLM (generated_results)**

📭 Таблица пуста.



### 💾 Таблица PostgreSQL: **Системные логи (logs)**

,id,timestamp,level,event,duration,context_size,message,traceback
0,1,2026-06-12 05:10:14.576453,INFO,document_indexing,46.80,19319,Успешная индексация файла Приказ МФ и НП N 84-...,None...
1,2,2026-06-12 05:10:16.172574,INFO,document_indexing,1.59,32471,Успешная индексация файла Регламент КФМ от 13....,None...
2,3,2026-06-12 05:10:19.617864,INFO,document_indexing,3.44,107538,Успешная индексация файла Приказ МФ и НП Новос...,None...
3,4,2026-06-12 05:10:20.715468,INFO,document_indexing,1.09,10776,Успешная индексация файла Постановление Правит...,None...
4,5,2026-06-12 05:10:22.065952,INFO,document_indexing,1.34,47093,Успешная индексация файла Постановление Правит...,None...
5,6,2026-06-12 05:10:29.113672,INFO,document_indexing,7.04,144036,Успешная индексация файла Приказ МФ и НП N 69...,None...
6,7,2026-06-12 05:10:36.086578,INFO,document_indexing,6.97,158534,Успешная индексация файла Приказ МФ и НП N 7-Н...,None...
7,8,2026-06-12 05:10:37.163202,INFO,document_indexing,1.07,7445,Успешная индексация файла Порядок.pdf.txt. Соз...,None...
8,9,2026-06-12 05:10:38.130912,INFO,document_indexing,0.96,2180,Успешная индексация файла Экспертиза исполните...,None...
9,10,2026-06-12 05:11:25.932783,INFO,document_indexing,47.80,1610973,Успешная индексация файла Приказ Минфина Росси...,None...


### 🗃️ Таблица SQLRecordManager: **Кэш чанков (SQLite)**

,uuid,key,namespace,group_id,updated_at
0,4f4ba6d3-cc03-49d4-bbfe-112bceb8b6be,3c387534-17f8-5e10-ac99-58dc5877bdd5,rag_bpmn,a7cea93f5cdd0127e356c2a7f9f3eaa2fd1cffca866e48...,1.781241e+09
1,3a207ab9-0393-487b-af09-21d1b941e8cf,822a7bac-bda2-57e2-aedc-24ccd7c7fbbd,rag_bpmn,a7cea93f5cdd0127e356c2a7f9f3eaa2fd1cffca866e48...,1.781241e+09
2,45fff6cd-e825-491e-8afb-3654de056dcf,c8796a70-2ff8-5fac-bac0-8178e6078d79,rag_bpmn,a7cea93f5cdd0127e356c2a7f9f3eaa2fd1cffca866e48...,1.781241e+09
3,ec83459f-b632-4c86-ac2b-498721fc3174,1dad37f3-8cf2-5d49-83fb-80f4d02815d2,rag_bpmn,a7cea93f5cdd0127e356c2a7f9f3eaa2fd1cffca866e48...,1.781241e+09
4,71882062-f7b1-4b52-8c80-6e5c9b5ecbcb,1c4f48ce-a029-588a-95fe-4a5d2c3bf5d6,rag_bpmn,a7cea93f5cdd0127e356c2a7f9f3eaa2fd1cffca866e48...,1.781241e+09
5,7fddcad8-905f-41c6-9199-74c8cc432f36,7478040d-6e2b-5480-a381-a6c78b539380,rag_bpmn,a7cea93f5cdd0127e356c2a7f9f3eaa2fd1cffca866e48...,1.781241e+09
6,b97d4369-a56c-44b6-ad03-62342520e9c4,ddc8ae7c-b7f2-5b1e-b2d8-9e891bb1ff55,rag_bpmn,a7cea93f5cdd0127e356c2a7f9f3eaa2fd1cffca866e48...,1.781241e+09
7,84b046f3-97de-4d3a-a442-3443e87c22d9,c427cbd0-413b-59d5-8689-6fd6b033207f,rag_bpmn,a7cea93f5cdd0127e356c2a7f9f3eaa2fd1cffca866e48...,1.781241e+09
8,5eed0065-eaee-4cb0-ac64-a9729dafdc83,99ef6f79-ac05-5a90-a0d0-8a1518c7062f,rag_bpmn,a7cea93f5cdd0127e356c2a7f9f3eaa2fd1cffca866e48...,1.781241e+09
9,4e1fc4db-f048-4028-abf4-174b02de227b,7036b2e3-f96f-570f-a2b8-7ff22f19bd85,rag_bpmn,a7cea93f5cdd0127e356c2a7f9f3eaa2fd1cffca866e48...,1.781241e+09


📊 Всего уникальных записей (чанков) под управлением менеджера: 1658


In [ ]:
import xml.dom.minidom

row = await database.fetch_one(xml_documents_table.select().limit(2))
if row:
    xml_string = row['xml_content']
    # Pretty print the XML content
    dom = xml.dom.minidom.parseString(xml_string)
    pretty_xml = dom.toprettyxml(indent="  ")
    print(pretty_xml)
else:
    print("Таблица 'xml_documents' пуста.")

<?xml version="1.0" ?>
<BPMN_Logical_Structure>
  <FlowElement id="umNi5orVADHmK-9Et_Aa-8" name="Отраслевые кураторы"/>
  <FlowElement id="NN1y8VhjrhkhN1qH7-dR-1" name="Поступление предложенияк изменению перечня"/>
  <SequenceFlow id="NN1y8VhjrhkhN1qH7-dR-9" sourceRef="NN1y8VhjrhkhN1qH7-dR-2" targetRef="NN1y8VhjrhkhN1qH7-dR-8"/>
  <FlowElement id="NN1y8VhjrhkhN1qH7-dR-2" name="Провести анализ на обоснованность"/>
  <SequenceFlow id="NN1y8VhjrhkhN1qH7-dR-3" sourceRef="NN1y8VhjrhkhN1qH7-dR-1" targetRef="NN1y8VhjrhkhN1qH7-dR-2"/>
  <SequenceFlow id="NN1y8VhjrhkhN1qH7-dR-10" sourceRef="NN1y8VhjrhkhN1qH7-dR-8" targetRef="NN1y8VhjrhkhN1qH7-dR-102"/>
  <FlowElement id="NN1y8VhjrhkhN1qH7-dR-11" name="Да"/>
  <SequenceFlow id="NN1y8VhjrhkhN1qH7-dR-13" sourceRef="NN1y8VhjrhkhN1qH7-dR-8" targetRef="NN1y8VhjrhkhN1qH7-dR-100"/>
  <FlowElement id="NN1y8VhjrhkhN1qH7-dR-14" name="Нет"/>
  <FlowElement id="NN1y8VhjrhkhN1qH7-dR-8" name="Согласовано"/>
  <SequenceFlow id="NN1y8VhjrhkhN1qH7-dR-22" sourceR

In [ ]:
async def show_full_prompts():
    """Выводит полное содержание ячейки full_prompt из таблицы generated_results."""
    print("\n=== Полное содержание 'full_prompt' из таблицы 'generated_results' ===\n")

    # Выбираем только столбец 'full_prompt'
    query = select(generated_results_table.c.full_prompt)
    rows = await database.fetch_all(query)

    if not rows:
        print("📭 Таблица 'generated_results' пуста или не содержит данных в 'full_prompt'.")
        return

    for i, row in enumerate(rows):
        prompt_content = row['full_prompt']
        if prompt_content:
            display(Markdown(f"### Промпт {i + 1}:\n```\n{prompt_content}\n```"))
        else:
            display(Markdown(f"### Промпт {i + 1}: (пусто)"))
        print("\n" + "=" * 80 + "\n") # Разделитель для читаемости

await show_full_prompts()

In [ ]:
# @title Очистить таблицы

# 2. Очищаем таблицы в Postgres
await database.execute(xml_documents_table.delete())
await database.execute(logs_table.delete())

# 3. Безопасно удаляем файл кэша SQLite, если он существует
import os
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
    print(f"✓ Файл кэша Record Manager ({DB_PATH}) успешно удален с диска.")
else:
    print("ℹ Файл кэша SQLite еще не был создан на диске, удаление не требуется.")

print("🚀 Базы данных полностью очищены и готовы к чистому повторному прогону!")

✓ Файл кэша Record Manager (/content/Project/data/db_file/record_manager_cache.sql) успешно удален с диска.
🚀 Базы данных полностью очищены и готовы к чистому повторному прогону!


## XML / JSON - генерация

### Вспомогательные функции

* Сборка контекста - собирает эталонные схемы из Postgres для отправки в контекст нейросети.
* Экспорт результатов в файл
* Внутренние функции для generate_bpmn_xml и generate_process_json

In [ ]:
# @title Сборка контекста XML

async def get_all_reference_xmls(docs: list) -> str:
    """Находит уникальные XML в Postgres по метаданным чанков и собирает единый контекст."""
    unique_parent_ids = list(set(d.metadata.get("parent_doc_id") for d in docs if d.metadata.get("parent_doc_id")))

    if not unique_parent_ids:
        return ""

    query = xml_documents_table.select().where(xml_documents_table.c.parent_doc_id.in_(unique_parent_ids))
    db_results = await database.fetch_all(query)

    xml_context = "\n\n=== СПИСОК ЭТАЛОННЫХ XML ДЛЯ КОНТЕКСТА ===\n"
    for row in db_results:
        xml_context += f"\nФайл: {row['filename']} (ID: {row['parent_doc_id']})\n"
        xml_context += f"Содержимое XML:\n{row['xml_content']}\n"
        xml_context += "-" * 30 + "\n"

    return xml_context

print("✓ Функциz сборки XML-контекста успешно загружена в память сессии.")


✓ Функциz сборки XML-контекста успешно загружена в память сессии.


In [ ]:
# @title  Экспорт результатов в файл

async def export_result_to_file(result_id: str, output_folder: str = "/content/TUSUR_Project/data/exports"):
    """
    Находит сгенерированный ответ в PostgreSQL по его result_id,
    автоматически определяет тип (xml/json) и сохраняет как готовый файл на Диск.
    """
    if not result_id:
        print("⚠️ Ошибка: Передан пустой result_id.")
        return None

    try:
        # 1. Запрашиваем сгенерированный контент из базы
        query = generated_results_table.select().where(generated_results_table.c.id == result_id)
        row = await database.fetch_one(query)

        if not row:
            print(f"❌ Запись с ID {result_id} не найдена в таблице generated_results.")
            return None

        # 2. Извлекаем данные из строки
        result_type = row["result_type"]
        raw_content = row["content"]

        # Создаем имя файла на основе ID и типа результата
        filename = f"generation_result_{str(result_id)[:8]}.{result_type}"

        # Создаем папку для экспорта, если её ещё нет
        os.makedirs(output_folder, exist_ok=True)
        full_file_path = os.path.join(output_folder, filename)

        # 3. Форматируем и сохраняем контент в зависимости от типа
        if result_type == "json":
            try:
                # Если в контенте лежит строка JSON, попробуем её красиво отформатировать с отступами
                # На случай, если Qwen выдал JSON внутри markdown-тегов ```json ... ```, очистим их
                clean_json_text = re.sub(r"```json\s*|```", "", raw_content).strip()
                json_data = json.loads(clean_json_text)
                with open(full_file_path, "w", encoding="utf-8") as f:
                    json.dump(json_data, f, ensure_ascii=False, indent=4)
            except Exception:
                # Если это не строгий JSON, а просто текст, сохраняем как есть
                with open(full_file_path, "w", encoding="utf-8") as f:
                    f.write(raw_content)
        else:
            # Для XML (BPMN-схем) очищаем markdown-обёртку ```xml ... ``` перед сохранением
            clean_xml_text = re.sub(r"```xml\s*|```", "", raw_content).strip()
            with open(full_file_path, "w", encoding="utf-8") as f:
                f.write(clean_xml_text)

        print(f"💾 Файл успешно экспортирован!")
        print(f"📂 Путь на Google Диске: {full_file_path}")
        return full_file_path

    except Exception as e:
        print(f"💥 Ошибка при экспорте файла: {str(e)}")
        return None


In [ ]:
# @title Внутренние функции для generate_bpmn_xml и generate_process_json

def _display_colab_ui(content: str, res_type: str, result_id: uuid.UUID):
    """Красиво выводит маркдаун ответа и голубую плашку ID на экран."""
    print("\n" + "="*60)
    display(Markdown(f"### 🎯 Ответ модели Qwen ({res_type.upper()}):"))
    display(Markdown(content))
    print("="*60)
    id_html = f"""
    <div style="background-color: #e6f6ff; padding: 12px; border-radius: 6px; border: 1px solid #b3dbff; margin-top: 15px;">
        <span style="font-weight: bold; color: #0052cc;">🔑 ID генерации (result_id):</span>
        <code style="font-family: monospace; font-size: 14px; background-color: #ffffff; padding: 2px 6px; border-radius: 4px; border: 1px solid #dcdcdc; color: #333333;">{result_id}</code>
    </div>
    """
    display(HTML(id_html))
    print("\n")

async def _log_error_to_db(event_name: str, start_time: float, stack_trace: str):
    """Записывает упавшую транзакцию в системные логи Postgres."""
    duration = time.time() - start_time
    error_msg = stack_trace.splitlines()[-1]
    print(f"💥 Ошибка в {event_name}: {error_msg}")
    try:
        await database.execute(logs_table.insert().values(
            level="ERROR", event=event_name, duration=duration,
            context_size=0, message=error_msg, traceback=stack_trace
        ))
    except Exception as e:
        print(f"Не удалось записать лог ошибки: {e}")

## Интерактивная RAG-система (Инференс Qwen)

In [ ]:
# @title Подключаем наши внешние шаблоны

# убедитесь, что папка prompts перенесена в корень
try:
    from prompts.bpmn_xml_rules import template as xml_template
    from prompts.bpmn_table_rules import template as table_template
except ImportError:
    # Заглушки на случай, если файлы промптов еще не перенесены
    xml_template = "Вы — эксперт BPMN. Сгенерируй корректный BPMN XML на основе контекста:\n{context}"
    table_template = "Вы — аналитик. Сформируй JSON-таблицу сущностей на основе контекста:\n{context}"

In [ ]:
# @title  ФУНКЦИЯ 1: ГЕНЕРАЦИЯ BPMN XML (ГИБРИДНЫЙ RAG: ChromaDB + Postgres)

async def generate_bpmn_xml(query_text: str, document_path: str = None, max_context: int = None, max_input: int = None):
    """
    Генерирует BPMN XML на основе сфокусированного контекста:
    ChromaDB используется как фильтр для поиска релевантных документов,
    но в переменной context_text остаются ТОЛЬКО связанные XML-эталоны из Postgres (без текста чанков).
    """
    start_time = time.time()
    if not query_text.strip():
        print("⚠️ Ошибка: Запрос пуст.")
        return None

    current_max_context = max_context or MAX_CONTEXT_CHARS
    current_max_input = max_input or MAX_INPUT_CHARS
    extra_doc_content = ""
    parent_id = "unknown"

    try:
        # 1. Чтение внешнего документа (целевого регламента для обработки)
        if document_path:
            if not os.path.exists(document_path):
                raise FileNotFoundError(f"Файл не найден: {document_path}")
            print(f"📖 Чтение документа: {os.path.basename(document_path)}...")
            extra_doc_content = _read_local_document(document_path)

        # 2. ПОИСК И СБОРКА КОНТЕКСТА (Исключаем сырой текст чанков)
        print("🔍 [XML-Режим] Поиск релевантных контекстов в ChromaDB...")
        combined_query = query_text
        if extra_doc_content:
            combined_query = f"{query_text} {extra_doc_content[:9000]}"

        # Делаем семантический поиск по векторной базе
        docs = await chroma.asimilarity_search(combined_query, k=7)

        if docs:
            parent_id = docs[0].metadata.get("parent_doc_id", "unknown")

            # ИСПРАВЛЕНО: Мы больше НЕ пишем `context_text = "\n\n".join([d.page_content for d in docs])`
            # Вместо этого мы сразу вызываем Postgres, оставляя в контексте только чистый XML-код схем
            print("💾 [XML-Режим] Текстовые чанки ChromaDB отброшены. Извлечение связанных XML-эталонов из Postgres...")
            context_text = await get_all_reference_xmls(docs)

            if not context_text.strip():
                print("⚠️ Предупреждение: Связанные XML-схемы не найдены в Postgres.")
                context_text = "XML-файлы эталонов отсутствуют в базе данных."
        else:
            parent_id = "unknown"
            print("⚠️ Предупреждение: В ChromaDB не найдено подходящих документов.")
            context_text = "XML-файлы эталонов не найдены."

        # Ограничиваем размер контекста по лимитам из констант
        if len(context_text) > current_max_context:
            context_text = context_text[:current_max_context]

        # 3. Человеческий ввод (User Input)
        full_input = query_text
        if extra_doc_content:
            full_input = f"ИНСТРУКЦИЯ КЛИЕНТА: {query_text}\n\nТЕКСТ ДОКУМЕНТА ДЛЯ ОБРАБОТКИ:\n{extra_doc_content}"
        if len(full_input) > current_max_input:
            full_input = full_input[:current_max_input]

        # 4. Сборка промпта
        chat_prompt = ChatPromptTemplate.from_messages([
            ("system", xml_template),
            ("human", "{input}"),
        ])
        final_prompt_text = chat_prompt.format(context=context_text, input=full_input)

        # Читаем параметры: stop берем из модели, а num_predict — фиксировано из параметров вызова
        model_m_kwargs = getattr(llm, "model_kwargs", {})
        # Проверяем сначала внутренний словарь, затем корень объекта, если везде пусто — возвращаем []
        stop_words = model_m_kwargs.get("stop") or getattr(llm, "stop", [])

        print(
            f"🤖 Отправка запроса в Ollama:\n"
            f"   • Модель: {getattr(llm, 'model', 'unknown')}\n"
            f"   • Контекст (num_ctx): {getattr(llm, 'num_ctx', 'default')} токенов\n"
            f"   • Температура: {getattr(llm, 'temperature', 'default')}\n"
            f"   • Лимит ответа (num_predict): 3000 токенов (вынесен в функцию)\n"  # Отображаем реальный лимит
            f"   • Стоп-слова: {stop_words}\n"
            f"   • Режим работы: XML"
        )
        # Сохраняем цепочку
        chain = chat_prompt | llm

        # ИСПРАВЛЕНО: Лимит передается напрямую в вызов асинхронной цепочки через config
        response = await chain.ainvoke(
            {"context": context_text, "input": full_input},
            config={"options": {"num_predict": 3000}}  # <-- Гарантированный проброс лимита в Ollama
        )

        raw_xml = response.content.strip()


        # =====================================================================
        # БЛОК АВТОМАТИЧЕСКОЙ ВАЛИДАЦИИ И ОЧИСТКИ XML
        # =====================================================================
        print("⚙️ Запуск автоматического валидатора тегов...")
        start_tag_match = re.search(r"<", raw_xml)
        if start_tag_match:
            raw_xml = raw_xml[start_tag_match.start():].strip()

        raw_xml = re.sub(r"```xml\s*|```", "", raw_xml).strip()

        is_valid_xml = True
        xml_error_message = ""
        try:
            ET.fromstring(raw_xml)
        except ET.ParseError as parse_err:
            is_valid_xml = False
            xml_error_message = str(parse_err)

        # =====================================================================
        # ОБРАБОТКА РЕЗУЛЬТАТОВ ВАЛИДАЦИИ И ЗАПИСЬ
        # =====================================================================
        new_result_id = uuid.uuid4()
        current_time_utc = datetime.now(timezone.utc).replace(tzinfo=None)

        if is_valid_xml:
            print("✅ Валидация успешна! Структура BPMN XML полностью корректна.")
            await database.execute(generated_results_table.insert().values(
                id=new_result_id, parent_doc_id=parent_id, result_type="xml",
                full_prompt=final_prompt_text, content=raw_xml, created_at=current_time_utc
            ))

            await database.execute(logs_table.insert().values(
                level="INFO", event="xml_generation", duration=time.time() - start_time,
                context_size=len(context_text), message=f"Успешная генерация валидного XML (Без чанков). ID: {new_result_id}"
            ))

            _display_colab_ui(raw_xml, "xml", new_result_id)
            return {"result_id": new_result_id, "type": "xml", "parent_doc_id": parent_id}

        else:
            print(f"❌ ВНИМАНИЕ: Модель сгенерировала СЛОМАННЫЙ XML-код!")
            print(f"🛑 Ошибка парсера: {xml_error_message}")

            await database.execute(logs_table.insert().values(
                level="ERROR", event="xml_validation_error", duration=time.time() - start_time,
                context_size=len(context_text), message=f"Битый XML от модели. Ошибка: {xml_error_message}",
                traceback=raw_xml
            ))

            display(Markdown("### ⚠️ Сырой некорректный ответ модели (не сохранен в xml_documents):"))
            print(raw_xml)
            return None

    except Exception as e:
        await _log_error_to_db("xml_generation", start_time, traceback.format_exc())
        return None

In [ ]:
# @title ФУНКЦИЯ 2: ГЕНЕРАЦИЯ СТРУКТУРЫ JSON

async def generate_process_json(query_text: str, document_path: str = None, max_context: int = None, max_input: int = None):
    """
    Генерирует JSON структуру процессов на основе XML-эталонов из Postgres.
    Автоматически очищает, форматирует (Pretty Print) и валидирует полученный JSON.
    """
    start_time = time.time()
    if not query_text.strip():
        print("⚠️ Ошибка: Запрос пуст.")
        return None

    current_max_context = max_context or MAX_CONTEXT_CHARS
    current_max_input = max_input or MAX_INPUT_CHARS
    extra_doc_content = ""
    parent_id = "unknown"

    try:
        # 1. Чтение внешнего обрабатываемого документа
        if document_path:
            if not os.path.exists(document_path):
                raise FileNotFoundError(f"Файл не найден: {document_path}")
            print(f"📖 Чтение документа: {os.path.basename(document_path)}...")
            extra_doc_content = _read_local_document(document_path)

        # 2. РЕЖИМ RAG ДЛЯ JSON: Извлекаем связанные XML-эталоны через ChromaDB
        print("🔍 [JSON-Режим] Поиск релевантных контекстов в ChromaDB...")
        combined_query = query_text
        if extra_doc_content:
            combined_query = f"{query_text} {extra_doc_content[:9000]}"

        docs = await chroma.asimilarity_search(combined_query, k=7)

        if docs:
            parent_id = docs[0].metadata.get("parent_doc_id", "unknown") if hasattr(docs[0], 'metadata') else "unknown"
            print("💾 [JSON-Режим] Текстовые чанки отброшены. Извлечение СВЯЗАННЫХ XML-эталонов из Postgres...")
            context_text = await get_all_reference_xmls(docs)

            if not context_text.strip():
                print("⚠️ Предупреждение: Связанные XML-схемы не найдены в Postgres.")
                context_text = "Связанные XML-файлы контекста отсутствуют."
        else:
            print("⚠️ Предупреждение: В ChromaDB не найдено ни одного подходящего чанка.")
            context_text = "Релевантные XML-файлы контекста не найдены."

        if len(context_text) > current_max_context:
            context_text = context_text[:current_max_context]

        # 3. Человеческий ввод (User Input)
        full_input = query_text
        if extra_doc_content:
            full_input = f"ИНСТРУКЦИЯ КЛИЕНТА: {query_text}\n\nТЕКСТ ДОКУМЕНТА ДЛЯ ОБРАБОТКИ:\n{extra_doc_content}"
        if len(full_input) > current_max_input:
            full_input = full_input[:current_max_input]

        # 4. Обработка промпта и защита от KeyError (экранирование скобок JSON-примеров)
        safe_template = table_template.replace("{context}", "__CONTEXT_KEY__").replace("{input}", "__INPUT_KEY__")
        safe_template = safe_template.replace("{", "{{").replace("}", "}}")
        safe_template = safe_template.replace("__CONTEXT_KEY__", "{context}").replace("__INPUT_KEY__", "{input}")

        chat_prompt = ChatPromptTemplate.from_messages([
            ("system", safe_template),
            ("human", "{input}"),
        ])
        final_prompt_text = chat_prompt.format(context=context_text, input=full_input)

        # Читаем параметры: stop берем из модели, а num_predict — фиксировано из параметров вызова
        model_m_kwargs = getattr(llm, "model_kwargs", {})
        # Проверяем сначала внутренний словарь, затем корень объекта, если везде пусто — возвращаем []
        stop_words = model_m_kwargs.get("stop") or getattr(llm, "stop", [])

        print(
            f"🤖 Отправка запроса в Ollama:\n"
            f"   • Модель: {getattr(llm, 'model', 'unknown')}\n"
            f"   • Контекст (num_ctx): {getattr(llm, 'num_ctx', 'default')} токенов\n"
            f"   • Температура: {getattr(llm, 'temperature', 'default')}\n"
            f"   • Лимит ответа (num_predict): 3000 токенов (вынесен в функцию)\n"  # Отображаем реальный лимит
            f"   • Стоп-слова: {stop_words}\n"
            f"   • Режим работы: JSON"
        )

        # Сохраняем цепочку
        chain = chat_prompt | llm

        # ИСПРАВЛЕНО: Лимит передается напрямую в вызов асинхронной цепочки через config
        response = await chain.ainvoke(
            {"context": context_text, "input": full_input},
            config={"options": {"num_predict": 3000}}  # <-- Гарантированный проброс лимита в Ollama
        )

        raw_json = response.content.strip()


        # =====================================================================
        # БЛОК АВТОМАТИЧЕСКОЙ ОЧИСТКИ, ВАЛИДАЦИИ И PRETTY PRINT ДЛЯ JSON
        # =====================================================================
        print("⚙️ Запуск автоматического форматирования и валидации JSON...")

        # Шаг А: Срезаем текстовые вступления модели, находя первую открывающую скобку { или [
        start_bracket_match = re.search(r"\{|\[", raw_json)
        if start_bracket_match:
            raw_json = raw_json[start_bracket_match.start():].strip()

        # Шаг Б: Срезаем любые текстовые хвосты после закрывающей скобки } или ]
        end_bracket_match = list(re.finditer(r"\}|\]", raw_json))
        if end_bracket_match:
            raw_json = raw_json[:end_bracket_match[-1].end()].strip()

        # Шаг В: Удаляем маркдаун-обертки ```json или ```, если модель их добавила
        raw_json = re.sub(r"```json\s*|```", "", raw_json).strip()

        # Шаг Г: Парсим через библиотеку json для проверки структуры и форматирования с отступами
        is_valid_json = True
        json_error_message = ""
        pretty_json_content = raw_json

        try:
            parsed_json_object = json.loads(raw_json)
            # Переводим в строку с красивыми отступами в 4 пробела и поддержкой кириллицы
            pretty_json_content = json.dumps(parsed_json_object, ensure_ascii=False, indent=4)
        except json.JSONDecodeError as json_err:
            is_valid_json = False
            json_error_message = str(json_err)

        # =====================================================================
        # ЗАПИСЬ РЕЗУЛЬТАТОВ И ВЫВОД НА ЭКРАН
        # =====================================================================
        new_result_id = uuid.uuid4()
        current_time_utc = datetime.now(timezone.utc).replace(tzinfo=None)

        if is_valid_json:
            print("✅ JSON успешно валидирован и отформатирован!")

            # Сохраняем в базу данных красивый, структурированный текст
            await database.execute(generated_results_table.insert().values(
                id=new_result_id, parent_doc_id=str(parent_id), result_type="json",
                full_prompt=final_prompt_text, content=pretty_json_content, created_at=current_time_utc
            ))

            await database.execute(logs_table.insert().values(
                level="INFO", event="json_generation", duration=time.time() - start_time,
                context_size=len(context_text), message=f"Успешный Pretty Print JSON. ID: {new_result_id}"
            ))

            # Выводим на экран как блок кода для удобного копирования
            print("\n" + "="*60)
            display(Markdown(f"### 🎯 Ответ модели Qwen (Отформатированный JSON):"))
            display(Markdown(f"```json\n{pretty_json_content}\n```"))
            print("="*60)

            # Вывод голубой плашки ID
            id_html = f"""
            <div style="background-color: #e6f6ff; padding: 12px; border-radius: 6px; border: 1px solid #b3dbff; margin-top: 15px;">
                <span style="font-weight: bold; color: #0052cc;">🔑 ID генерации (result_id):</span>
                <code style="font-family: monospace; font-size: 14px; background-color: #ffffff; padding: 2px 6px; border-radius: 4px; border: 1px solid #dcdcdc; color: #333333;">{new_result_id}</code>
            </div>
            """
            display(HTML(id_html))
            print("\n")

            return {"result_id": new_result_id, "type": "json", "parent_doc_id": parent_id}
        else:
            print(f"❌ ВНИМАНИЕ: Модель выдала некорректную структуру JSON!")
            print(f"🛑 Ошибка парсера: {json_error_message}")

            # Фиксируем битый JSON в логи для последующего исправления промпта
            await database.execute(logs_table.insert().values(
                level="ERROR", event="json_validation_error", duration=time.time() - start_time,
                context_size=len(context_text), message=f"Битый JSON от модели. Ошибка: {json_error_message}",
                traceback=raw_json
            ))

            display(Markdown("### ⚠️ Некорректный ответ модели (не отформатирован):"))
            print(raw_json)
            return None

    except Exception as e:
        await _log_error_to_db("json_generation", start_time, traceback.format_exc())
        return None

## Запуск RAG-генерации

In [ ]:
# Путь к локальному документу, на основе которого ИИ будет строить модели
# (Может быть .txt, .pdf или .docx, лежащий на Google Диске или в Colab)
TARGET_DOCUMENT = "/content/drive/MyDrive/Colab Notebooks/RAG_BPMN/Экспертиза НПА/Крошина Д.Е. Проведение экспертизы _доработанная_.pdf"

print("============================================================")
print("🚀 ЭТАП 1: ГЕНЕРАЦИЯ СХЕМЫ BPMN (В ФОРМАТЕ XML)")
print("============================================================")

# Шаг 1: Задаем вопрос нейросети (слово "xml" активирует BPMN-шаблон)
xml_query = "Сгенерируй структуру xml для процесса экспертизы правовых актов."

# Шаг 2: Запускаем RAG-запрос с подключением внешнего файла
# Система найдет похожие чанки в ChromaDB, вытащит эталоны из Postgres,
# объединит их с текстом TARGET_DOCUMENT и отправит в Qwen
rag_xml_info = await generate_bpmn_xml(
    query_text=xml_query,
    document_path=TARGET_DOCUMENT
)

# Шаг 3: Автоматический экспорт сгенерированного XML на Google Диск
if rag_xml_info and rag_xml_info["result_id"]:
    exported_xml_path = await export_result_to_file(
        result_id=rag_xml_info["result_id"],
        output_folder="/content/drive/MyDrive/Colab Notebooks/RAG_BPMN/Экспертиза НПА"
    )


print("\n" + "="*60 + "\n")


print("============================================================")
print("🚀 ЭТАП 2: ГЕНЕРАЦИЯ СТРУКТУРЫ ПРОЦЕССА (В ФОРМАТЕ JSON-ТАБЛИЦЫ)")
print("============================================================")

# Шаг 1: Формируем запрос без слова 'xml' (это активирует шаблон таблицы сущностей)
table_query = "Выдели ключевые шаги, участников, условия и сущности из документа загруженного пользователем."

# Шаг 2: Запускаем RAG-запрос для построения текстовой структуры
rag_table_info = await generate_process_json(
    query_text=table_query,
    document_path=TARGET_DOCUMENT
)

# Шаг 3: Автоматический экспорт сгенерированной JSON-структуры на Google Диск
if rag_table_info and rag_table_info["result_id"]:
    exported_json_path = await export_result_to_file(
        result_id=rag_table_info["result_id"],
        output_folder="/content/drive/MyDrive/Colab Notebooks/RAG_BPMN/Экспертиза НПА"
    )


🚀 ЭТАП 1: ГЕНЕРАЦИЯ СХЕМЫ BPMN (В ФОРМАТЕ XML)
📖 Чтение документа: Крошина Д.Е. Проведение экспертизы _доработанная_.pdf...
🔍 [XML-Режим] Поиск релевантных контекстов в ChromaDB...
💾 [XML-Режим] Текстовые чанки ChromaDB отброшены. Извлечение связанных XML-эталонов из Postgres...
🤖 Отправка запроса в Ollama:
   • Модель: qwen2.5:14b-instruct-q4_K_M
   • Контекст (num_ctx): 32000 токенов
   • Температура: 0.25
   • Лимит ответа (num_predict): 3000 токенов (вынесен в функцию)
   • Стоп-слова: []
   • Режим работы: XML
⚙️ Запуск автоматического валидатора тегов...
✅ Валидация успешна! Структура BPMN XML полностью корректна.



### 🎯 Ответ модели Qwen (XML):

<BPMN_Logical_Structure>
    <FlowElement id="startEvent" name="Представление проекта правового акта">
        <LaneSet>
            <Lane id="lane1" name="Разработчик проекта"/>
            <Lane id="lane2" name="Юридическая служба"/>
            <Lane id="lane3" name="Правовое управление"/>
        </LaneSet>
    </FlowElement>
    <FlowElement id="task1" name="Представление проекта правового акта в юридическую службу">
        <LaneRef>lane1</LaneRef>
    </FlowElement>
    <SequenceFlow id="flow1" sourceRef="startEvent" targetRef="task1"/>
    <ExclusiveGateway id="exclusiveGateway1" name="Согласование проекта правовым управлением">
        <LaneRef>lane2</LaneRef>
    </ExclusiveGateway>
    <SequenceFlow id="flow2" sourceRef="task1" targetRef="exclusiveGateway1"/>
    <FlowElement id="task2" name="Согласование проекта правовым управлением">
        <LaneRef>lane3</LaneRef>
    </FlowElement>
    <SequenceFlow id="flow3" sourceRef="exclusiveGateway1" targetRef="task2" name="Да"/>
    <ExclusiveGateway id="exclusiveGateway2" name="Согласование проекта правовым управлением">
        <LaneRef>lane2</LaneRef>
    </ExclusiveGateway>
    <SequenceFlow id="flow4" sourceRef="exclusiveGateway1" targetRef="exclusiveGateway2" name="Нет"/>
    <FlowElement id="task3" name="Регистрация проекта в Журнале входящих документов">
        <LaneRef>lane1</LaneRef>
    </FlowElement>
    <SequenceFlow id="flow5" sourceRef="exclusiveGateway2" targetRef="task3"/>
    <ExclusiveGateway id="exclusiveGateway3" name="Экспертиза проектов правовых актов, разрабатываемых министерством финансов и налоговой политики Новосибирской области">
        <LaneRef>lane2</LaneRef>
    </ExclusiveGateway>
    <SequenceFlow id="flow6" sourceRef="task3" targetRef="exclusiveGateway3"/>
    <FlowElement id="task4" name="Размещение проекта НПА на независимую антикоррупционную экспертизу">
        <LaneRef>lane1</LaneRef>
    </FlowElement>
    <SequenceFlow id="flow7" sourceRef="exclusiveGateway3" targetRef="task4"/>
    <ExclusiveGateway id="exclusiveGateway4" name="Проведение экспертизы">
        <LaneRef>lane2</LaneRef>
    </ExclusiveGateway>
    <SequenceFlow id="flow8" sourceRef="task4" targetRef="exclusiveGateway4"/>
    <FlowElement id="task5" name="Правовая экспертиза">
        <LaneRef>lane2</LaneRef>
    </FlowElement>
    <SequenceFlow id="flow9" sourceRef="exclusiveGateway4" targetRef="task5"/>
    <ExclusiveGateway id="exclusiveGateway5" name="Антикоррупционная экспертиза (только в отношении проектов НПА)">
        <LaneRef>lane2</LaneRef>
    </ExclusiveGateway>
    <SequenceFlow id="flow10" sourceRef="task5" targetRef="exclusiveGateway5"/>
    <FlowElement id="task6" name="Юридико-техническая экспертиза">
        <LaneRef>lane2</LaneRef>
    </FlowElement>
    <SequenceFlow id="flow11" sourceRef="exclusiveGateway5" targetRef="task6"/>
    <ExclusiveGateway id="exclusiveGateway6" name="Лингвистическая экспертиза">
        <LaneRef>lane2</LaneRef>
    </ExclusiveGateway>
    <SequenceFlow id="flow12" sourceRef="task6" targetRef="exclusiveGateway6"/>
    <FlowElement id="task7" name="Фиксация хода ведения экспертизы">
        <LaneRef>lane2</LaneRef>
    </FlowElement>
    <SequenceFlow id="flow13" sourceRef="exclusiveGateway6" targetRef="task7"/>
    <ExclusiveGateway id="exclusiveGateway7" name="Направление проекта НПА на согласование в прокуратуру НСО">
        <LaneRef>lane2</LaneRef>
    </ExclusiveGateway>
    <SequenceFlow id="flow14" sourceRef="task7" targetRef="exclusiveGateway7"/>
    <FlowElement id="endEvent" name="Завершение процесса экспертизы правовых актов">
        <LaneRef>lane2</LaneRef>
    </FlowElement>
    <SequenceFlow id="flow15" sourceRef="exclusiveGateway7" targetRef="endEvent"/>
</BPMN_Logical_Structure>



💾 Файл успешно экспортирован!
📂 Путь на Google Диске: /content/drive/MyDrive/Colab Notebooks/RAG_BPMN/Экспертиза НПА/generation_result_65197e24.xml


🚀 ЭТАП 2: ГЕНЕРАЦИЯ СТРУКТУРЫ ПРОЦЕССА (В ФОРМАТЕ JSON-ТАБЛИЦЫ)
📖 Чтение документа: Крошина Д.Е. Проведение экспертизы _доработанная_.pdf...
🔍 [JSON-Режим] Поиск релевантных контекстов в ChromaDB...
💾 [JSON-Режим] Текстовые чанки отброшены. Извлечение СВЯЗАННЫХ XML-эталонов из Postgres...
🤖 Отправка запроса в Ollama:
   • Модель: qwen2.5:14b-instruct-q4_K_M
   • Контекст (num_ctx): 32000 токенов
   • Температура: 0.25
   • Лимит ответа (num_predict): 3000 токенов (вынесен в функцию)
   • Стоп-слова: []
   • Режим работы: JSON
⚙️ Запуск автоматического форматирования и валидации JSON...
💥 Ошибка в json_generation: UnboundLocalError: cannot access local variable 'raw_json' where it is not associated with a value


In [ ]:
print("============================================================")
print("🚀 ЭТАП 2: ГЕНЕРАЦИЯ СТРУКТУРЫ ПРОЦЕССА (В ФОРМАТЕ JSON-ТАБЛИЦЫ)")
print("============================================================")

# Шаг 1: Формируем запрос без слова 'xml' (это активирует шаблон таблицы сущностей)
table_query = "Выдели ключевые шаги, участников, условия и сущности из документа загруженного пользователем."

# Шаг 2: Запускаем RAG-запрос для построения текстовой структуры
rag_table_info = await generate_process_json(
    query_text=table_query,
    document_path=TARGET_DOCUMENT
)

# Шаг 3: Автоматический экспорт сгенерированной JSON-структуры на Google Диск
if rag_table_info and rag_table_info["result_id"]:
    exported_json_path = await export_result_to_file(
        result_id=rag_table_info["result_id"],
        output_folder="/content/drive/MyDrive/Colab Notebooks/RAG_BPMN/Экспертиза НПА"
    )

🚀 ЭТАП 2: ГЕНЕРАЦИЯ СТРУКТУРЫ ПРОЦЕССА (В ФОРМАТЕ JSON-ТАБЛИЦЫ)
📖 Чтение документа: Крошина Д.Е. Проведение экспертизы _доработанная_.pdf...
🔍 [JSON-Режим] Поиск релевантных контекстов в ChromaDB...
💾 [JSON-Режим] Текстовые чанки отброшены. Извлечение СВЯЗАННЫХ XML-эталонов из Postgres...
🤖 Отправка запроса в Ollama:
   • Модель: qwen2.5:14b-instruct-q4_K_M
   • Контекст (num_ctx): 32000 токенов
   • Температура: 0.25
   • Лимит ответа (num_predict): 3000 токенов (вынесен в функцию)
   • Стоп-слова: None
   • Режим работы: JSON
⚙️ Запуск автоматического форматирования и валидации JSON...
✅ JSON успешно валидирован и отформатирован!



### 🎯 Ответ модели Qwen (Отформатированный JSON):

```json
[
    {
        "id": "start_01",
        "type": "startEvent",
        "name": "Представление проекта правового акта в юридическую службу"
    },
    {
        "id": "task_01",
        "type": "task",
        "role": "Разработчик проекта",
        "name": "Внесение сведений в Журнал входящих документов"
    },
    {
        "id": "gateway_01",
        "type": "exclusiveGateway",
        "name": "Проект правового акта - закон НСО?",
        "branches": {
            "Да": "task_02",
            "Нет": "task_03"
        }
    },
    {
        "id": "task_02",
        "type": "task",
        "role": "Юрист-эксперт, начальник юридической службы, начальник правового управления",
        "name": "Согласование проекта законов НСО"
    },
    {
        "id": "task_03",
        "type": "task",
        "role": "Юрист-эксперт, начальник юридической службы, начальник правового управления",
        "name": "Согласование проекта правовых актов министерства и Губернатора НСО/Правительства НСО"
    },
    {
        "id": "task_04",
        "type": "task",
        "role": "Юрист-эксперт",
        "name": "Проведение правовой экспертизы проектов правовых актов министерства финансов и налоговой политики НСО"
    },
    {
        "id": "gateway_02",
        "type": "exclusiveGateway",
        "name": "Проект НПА?",
        "branches": {
            "Да": "task_05",
            "Нет": "endEvent"
        }
    },
    {
        "id": "task_05",
        "type": "task",
        "role": "Юрист-эксперт",
        "name": "Размещение проекта НПА на независимую антикоррупционную экспертизу"
    },
    {
        "id": "task_06",
        "type": "task",
        "role": "Юрист-эксперт",
        "name": "Проведение антикоррупционной экспертизы проектов НПА"
    },
    {
        "id": "task_07",
        "type": "task",
        "role": "Юрист-эксперт",
        "name": "Юридико-техническая экспертиза"
    },
    {
        "id": "task_08",
        "type": "task",
        "role": "Лингвист",
        "name": "Лингвистическая экспертиза"
    },
    {
        "id": "task_09",
        "type": "task",
        "role": "Юрист-эксперт",
        "name": "Фиксация хода ведения экспертизы"
    },
    {
        "id": "gateway_03",
        "type": "exclusiveGateway",
        "name": "Проект НПА (не проект закона НСО)?",
        "branches": {
            "Да": "task_10",
            "Нет": "endEvent"
        }
    },
    {
        "id": "task_10",
        "type": "task",
        "role": "Юрист-эксперт",
        "name": "Направление проекта НПА на согласование в прокуратуру НСО"
    },
    {
        "id": "endEvent",
        "type": "endEvent",
        "name": "Завершение экспертизы правовых актов"
    }
]
```



💾 Файл успешно экспортирован!
📂 Путь на Google Диске: /content/drive/MyDrive/Colab Notebooks/RAG_BPMN/Экспертиза НПА/generation_result_b76d03b0.json


In [ ]:
# @title Просмотрщик сгенерированных результатов и промптов

async def inspect_generation_result(result_id: str):
    """
    Находит в PostgreSQL запись по result_id и выводит
    в читаемом виде ПОЛНЫЙ промпт и полученный ответ модели.
    """
    if not result_id:
        print("⚠️ Укажите корректный result_id.")
        return

    try:
        # 1. Запрашиваем строку из базы данных
        query = generated_results_table.select().where(generated_results_table.c.id == result_id)
        row = await database.fetch_one(query)

        if not row:
            print(f"❌ Запись с ID {result_id} не найдена в базе данных.")
            return

        print("=" * 70)
        print(f"🔍 АНАЛИЗ ГЕНЕРАЦИИ ДЛЯ ID: {result_id}")
        print(f"📅 Время создания: {row['created_at']} | Тип: {row['result_type'].upper()}")
        print("=" * 70)

        # 2. Выводим полный промпт в раскрывающемся спойлере (чтобы не занимал весь экран)
        prompt_html = f"""
        <details style="background-color: #f0f4f8; padding: 10px; border-radius: 5px; border: 1px solid #d0d7de;">
            <summary style="font-weight: bold; cursor: pointer; color: #0969da;">
                📋 Показать ПОЛНЫЙ промпт, отправленный в Ollama ({len(row['full_prompt'])} симв.)
            </summary>
            <pre style="white-space: pre-wrap; font-family: monospace; margin-top: 10px; color: #24292f;">{row['full_prompt']}</pre>
        </details>
        """
        display(HTML(prompt_html))
        print("\n")

        # 3. Выводим ответ модели в формате Markdown
        display(Markdown("### 🎯 Ответ модели, сохраненный в базе:"))
        display(Markdown(row['content']))
        print("=" * 70)

    except Exception as e:
        print(f"💥 Ошибка при чтении из базы данных: {e}")


In [ ]:
# ==========================================
# ПРИМЕР ВЫЗОВА (ПОДСТАВЬТЕ СВОЙ ID ПОСЛЕ ЗАПУСКА RAG)
# ==========================================

await inspect_generation_result("65197e24-da49-424b-b93e-0c48289ac726")


🔍 АНАЛИЗ ГЕНЕРАЦИИ ДЛЯ ID: 65197e24-da49-424b-b93e-0c48289ac726
📅 Время создания: 2026-06-13 02:50:09.370055 | Тип: XML


### 🎯 Ответ модели, сохраненный в базе:

<BPMN_Logical_Structure>
    <FlowElement id="startEvent" name="Представление проекта правового акта">
        <LaneSet>
            <Lane id="lane1" name="Разработчик проекта"/>
            <Lane id="lane2" name="Юридическая служба"/>
            <Lane id="lane3" name="Правовое управление"/>
        </LaneSet>
    </FlowElement>
    <FlowElement id="task1" name="Представление проекта правового акта в юридическую службу">
        <LaneRef>lane1</LaneRef>
    </FlowElement>
    <SequenceFlow id="flow1" sourceRef="startEvent" targetRef="task1"/>
    <ExclusiveGateway id="exclusiveGateway1" name="Согласование проекта правовым управлением">
        <LaneRef>lane2</LaneRef>
    </ExclusiveGateway>
    <SequenceFlow id="flow2" sourceRef="task1" targetRef="exclusiveGateway1"/>
    <FlowElement id="task2" name="Согласование проекта правовым управлением">
        <LaneRef>lane3</LaneRef>
    </FlowElement>
    <SequenceFlow id="flow3" sourceRef="exclusiveGateway1" targetRef="task2" name="Да"/>
    <ExclusiveGateway id="exclusiveGateway2" name="Согласование проекта правовым управлением">
        <LaneRef>lane2</LaneRef>
    </ExclusiveGateway>
    <SequenceFlow id="flow4" sourceRef="exclusiveGateway1" targetRef="exclusiveGateway2" name="Нет"/>
    <FlowElement id="task3" name="Регистрация проекта в Журнале входящих документов">
        <LaneRef>lane1</LaneRef>
    </FlowElement>
    <SequenceFlow id="flow5" sourceRef="exclusiveGateway2" targetRef="task3"/>
    <ExclusiveGateway id="exclusiveGateway3" name="Экспертиза проектов правовых актов, разрабатываемых министерством финансов и налоговой политики Новосибирской области">
        <LaneRef>lane2</LaneRef>
    </ExclusiveGateway>
    <SequenceFlow id="flow6" sourceRef="task3" targetRef="exclusiveGateway3"/>
    <FlowElement id="task4" name="Размещение проекта НПА на независимую антикоррупционную экспертизу">
        <LaneRef>lane1</LaneRef>
    </FlowElement>
    <SequenceFlow id="flow7" sourceRef="exclusiveGateway3" targetRef="task4"/>
    <ExclusiveGateway id="exclusiveGateway4" name="Проведение экспертизы">
        <LaneRef>lane2</LaneRef>
    </ExclusiveGateway>
    <SequenceFlow id="flow8" sourceRef="task4" targetRef="exclusiveGateway4"/>
    <FlowElement id="task5" name="Правовая экспертиза">
        <LaneRef>lane2</LaneRef>
    </FlowElement>
    <SequenceFlow id="flow9" sourceRef="exclusiveGateway4" targetRef="task5"/>
    <ExclusiveGateway id="exclusiveGateway5" name="Антикоррупционная экспертиза (только в отношении проектов НПА)">
        <LaneRef>lane2</LaneRef>
    </ExclusiveGateway>
    <SequenceFlow id="flow10" sourceRef="task5" targetRef="exclusiveGateway5"/>
    <FlowElement id="task6" name="Юридико-техническая экспертиза">
        <LaneRef>lane2</LaneRef>
    </FlowElement>
    <SequenceFlow id="flow11" sourceRef="exclusiveGateway5" targetRef="task6"/>
    <ExclusiveGateway id="exclusiveGateway6" name="Лингвистическая экспертиза">
        <LaneRef>lane2</LaneRef>
    </ExclusiveGateway>
    <SequenceFlow id="flow12" sourceRef="task6" targetRef="exclusiveGateway6"/>
    <FlowElement id="task7" name="Фиксация хода ведения экспертизы">
        <LaneRef>lane2</LaneRef>
    </FlowElement>
    <SequenceFlow id="flow13" sourceRef="exclusiveGateway6" targetRef="task7"/>
    <ExclusiveGateway id="exclusiveGateway7" name="Направление проекта НПА на согласование в прокуратуру НСО">
        <LaneRef>lane2</LaneRef>
    </ExclusiveGateway>
    <SequenceFlow id="flow14" sourceRef="task7" targetRef="exclusiveGateway7"/>
    <FlowElement id="endEvent" name="Завершение процесса экспертизы правовых актов">
        <LaneRef>lane2</LaneRef>
    </FlowElement>
    <SequenceFlow id="flow15" sourceRef="exclusiveGateway7" targetRef="endEvent"/>
</BPMN_Logical_Structure>

# Превращение логического XML/json в графический Draw.io

In [ ]:
# @title Автолейаут на основе BPMN XML



In [ ]:
if rag_xml_info:
    # Вытаскиваем только что сгенерированный чистый XML текст из Postgres
    query = generated_results_table.select().where(generated_results_table.c.id == rag_xml_info["result_id"])
    row = await database.fetch_one(query)

    # Запускаем наш Скрипт 1 для расчета координат и сборки .drawio файла!
    layout_bpmn_xml_to_drawio(
        xml_content_str=row["content"],
        output_path="/content/drive/MyDrive/Colab Notebooks/RAG_BPMN/Экспертиза НПА/BPMN_Diagram_from_XML.drawio"
    )


In [ ]:
# @title Автолейаут на основе JSON-структуры



In [ ]:
if rag_table_info:
    # Вытаскиваем сгенерированный чистый JSON текст из Postgres
    query = generated_results_table.select().where(generated_results_table.c.id == rag_table_info["result_id"])
    row = await database.fetch_one(query)

    # Запускаем наш Скрипт 2 для автоматической расстановки блоков таблицы по холсту!
    layout_json_structure_to_drawio(
        json_content_str=row["content"],
        output_path="/content/drive/MyDrive/Colab Notebooks/RAG_BPMN/Экспертиза НПА/Process_Flow_from_JSON.drawio"
    )
